# CMDuo Bulk Pop and WGD IPA analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib import colormaps
from matplotlib.colors import Normalize
from io import StringIO
from matplotlib.lines import Line2D
from matplotlib import transforms as mtransforms
from itertools import groupby
from collections import OrderedDict
import os
from matplotlib.colors import TwoSlopeNorm
from itertools import combinations

In [ ]:
class IPA:
    def __init__(self, filepath: str, molecules_filepath=None):
        self.filepath = filepath
        self.pathways = None
        self.upstream_regulators = None
        self.causal_networks = None
        self.regulator_effects = None
        self.analysis_molecules = None
        self.molecules = pd.DataFrame()
        
        self._parse_file()

        if molecules_filepath:
            self.add_molecules(molecules_filepath)

    def __repr__(self):
        return f'IPA Analysis Result with:\n\t{self.pathways.shape[0]} pathways\n\t{self.upstream_regulators.shape[0]} upstream regulators\n\t{self.causal_networks.shape[0]} causal networks\n\t{self.regulator_effects.shape[0]} regulator effects\n\t{self.analysis_molecules.shape[0]} analysis-ready molecules\n\t{self.molecules.shape[0]} molecules'

    def _extract_table(self, lines, start_header, end_condition):
        table_lines = []
        in_table = False

        for line in lines:
            if not in_table and line.startswith(start_header):
                in_table = True
                continue  # skip header line itself

            if in_table:
                if end_condition(line):
                    break
                table_lines.append(line)

        if not table_lines:
            return None

        table_str = "\n".join(table_lines).strip()
        df = pd.read_table(StringIO(table_str), sep="\t", index_col=False)
        df.columns = df.columns.str.strip()
        return df

    def _parse_file(self):
        with open(self.filepath, "r", encoding="utf-8", errors="ignore") as f:
            lines = [line.rstrip("\n") for line in f]

        try:
            self.pathways = self._extract_table(
                lines,
                "Canonical Pathways for",
                lambda l: l.strip() == ""
            )
            self.pathways = self.pathways.set_index('Ingenuity Canonical Pathways')
        except pd.errors.EmptyDataError:
            self.pathways = pd.DataFrame()

        try:
            self.upstream_regulators = self._extract_table(
                lines,
                "Upstream Regulators for",
                lambda l: l.strip() == ""
            )
        except pd.errors.EmptyDataError:
            self.upstream_regulators = pd.DataFrame()

        try:
            self.causal_networks = self._extract_table(
                lines,
                "Causal Networks for",
                lambda l: l.strip() == ""
            )
        except pd.errors.EmptyDataError:
            self.causal_networks = pd.DataFrame()

        try:
            self.regulator_effects = self._extract_table(
                lines,
                "Regulator Effects for",
                lambda l: l.startswith("Networks for")
            )
        except pd.errors.EmptyDataError:
            self.regulator_effects = pd.DataFrame()

        try:
            self.analysis_molecules = self._extract_table(
                lines,
                "Analysis Ready Molecules for",
                lambda l: l.strip() == ""
            )
        except pd.errors.EmptyDataError:
            self.analysis_molecules = pd.DataFrame()

    def add_molecules(self, filepath):
        self.molecules = pd.read_table(filepath, header=1, index_col=0)

    def get_pathways(self, min_p = 0., min_abs_z=None, min_z=None, max_z=None, min_ratio=0., search_genes_any=[], search_genes_all=[], non_nan=False, sort_values=None):
        mask = (self.pathways['-log(p-value)'] >= min_p) & \
            (self.pathways['Ratio'] >= min_ratio)

        if min_abs_z:
            mask = mask & (np.abs(self.pathways['zScore']) >= min_abs_z) 

        if min_z:
            mask = mask & (self.pathways['zScore'] >= min_z)

        if max_z:
            mask = mask & (self.pathways['zScore'] <= max_z)

        if non_nan:
            mask = mask & ~np.any(pd.isna(self.pathways), axis=1)

        if isinstance(search_genes_any, str):
            mask = mask & self.pathways['Molecules'].str.contains(search_genes_any)
        elif len(search_genes_any) > 0:
            np.any([self.pathways['Molecules'].str.contains(gene) for gene in search_genes_any], axis=0)

        if isinstance(search_genes_all, str):
            mask = mask & self.pathways['Molecules'].str.contains(search_genes_all)
        elif len(search_genes_all) > 0:
            np.all([self.pathways['Molecules'].str.contains(gene) for gene in search_genes_all], axis=0)

        if sort_values:
            return self.pathways.loc[mask, :].sort_values(sort_values, ascending=False)
        else:
            return self.pathways.loc[mask, :]
        


In [ ]:
def paths_barplot(paths_df, title=None, sig_line=True, cbar_height=0.3):
    fig, axs = plt.subplots(1, 2, figsize=(6, paths_df.shape[0]/4+0.2), gridspec_kw={'width_ratios': [20, 1]})

    ps = 10**(paths_df['-log(p-value)'])
    zs = paths_df['zScore']
    max_z = np.max(np.abs(zs))
    zs_n = (zs/max_z)/2+0.5

    axs[0].barh(paths_df.index, ps, color=[colormaps['coolwarm'](z) for z in zs_n], linewidth=1, edgecolor='black')

    if sig_line:    
        axs[0].vlines(1/0.05, paths_df.shape[0]+1, -2, linestyles='--', color='black')

    axs[0].set_xscale('log')
    axs[0].set_xlim(1, axs[0].get_xlim()[1])
    axs[0].set_xticklabels([l.get_text().replace('10^{', '10^{-') .replace('10^{-0}', '10^{0}') for l in axs[0].get_xticklabels()])
    axs[0].set_xlabel('$p$-value')

    axs[0].yaxis.set_inverted(True)
    axs[0].set_ylim(paths_df.shape[0], -1)

    if title:
        axs[0].set_title(title, fontweight='bold')

    fig.colorbar(ScalarMappable(norm=Normalize(vmin=-max_z, vmax=max_z), cmap='coolwarm'), cax=axs[1])
    
    cb_pos = axs[1].get_position()
    axs[1].set_position([cb_pos.x0-0.05, (cb_pos.y0+cb_pos.height/2)-(cbar_height/2), cb_pos.width, cbar_height])
    axs[1].set_title('$Z$-score', fontsize='medium', ha='left')
    #plt.tight_layout()

    return paths_df.index

## 231 Fusion Populations Comparisons

In [ ]:
files_231 = {
    "Fusion C1C4 vs MPV":  "/stor/work/Brock/kennedy/SC_repo/data/IPA_Analysis/231_matchedF_C1C4vsMPV_shrink.txt",
    "Fusion C6C8 vs MPV": "/stor/work/Brock/kennedy/SC_repo/data/IPA_Analysis/231_matchedF_C6C8vsMPV_shrink.txt",
}

results_231 = {label: IPA(fp) for label, fp in files_231.items()}

for label, ipa_obj in results_231.items():
    plot_df = ipa_obj.get_pathways(
        min_p=1.301, min_abs_z=2,non_nan=True, sort_values='-log(p-value)'
    ).head(50)

    print(label)
    print(plot_df)
    
    paths_barplot(plot_df, title=label)
    
    safe_label = label.replace(" ", "_").replace("/", "-")
    plt.savefig(f"pathways_231_{safe_label}.svg", bbox_inches='tight')
    plt.show()

## 1806 Fusion Population Comparisons

In [ ]:
files_1806 = {
    "Fusion C2C4 vs MPV":  "/stor/work/Brock/kennedy/SC_repo/data/IPA_Analysis/1806_matchedF_C2C4vsMPV_shrink.txt",
    "Fusion C5C2 vs MPV": "/stor/work/Brock/kennedy/SC_repo/data/IPA_Analysis/1806_matchedF_C5C2vsMPV_shrink.txt",
    "Fusion C6C7 vs MPV": "/stor/work/Brock/kennedy/SC_repo/data/IPA_Analysis/1806_matchedF_C6C7vsMPV_shrink.txt",

}

results_1806 = {label: IPA(fp) for label, fp in files_1806.items()}

for label, ipa_obj in results_1806.items():
    plot_df = ipa_obj.get_pathways(
        min_p=1.301, min_abs_z=2, non_nan=True, sort_values='-log(p-value)'
    ).head(50)

    print(label)
    print(plot_df)
    
    paths_barplot(plot_df, title=label)
    
    safe_label = label.replace(" ", "_").replace("/", "-")
    plt.savefig(f"pathways_1806_{safe_label}.svg", bbox_inches='tight')
    plt.show()

## Shared Pathway Dotplots

In [ ]:
def _resolve_pathway_name(name, index):
    """Fuzzy-match a pathway name against an IPA results index."""
    if name in index:
        return name
    low = index.str.lower()
    exact = index[low == name.lower()]
    if len(exact):
        return exact[0]
    sub = index[low.str.contains(name.lower(), regex=False)]
    return sub[0] if len(sub) else None


def _mean_neglogp(pathway, ipa_dict, comparisons, penalize_missing=True):
    """
    Mean -log10(p-value) for a single pathway across all comparisons.

    If penalize_missing=True (recommended default), clones where the pathway
    wasn't among IPA's reported top hits are treated as -log10(1) = 0 
    (i.e., no evidence of significance), so pathways lacking cross-clone
    recurrence are appropriately down-ranked rather than ignored.

    If penalize_missing=False, missing clones are simply excluded from the
    average (prior behavior).
    """
    vals = []
    for cl, key, _ in comparisons:
        candidates = [k for k in ipa_dict if cl in k and key in k]
        if not candidates:
            vals.append(0.0 if penalize_missing else None)
            continue
        df = ipa_dict[candidates[0]].pathways
        r = _resolve_pathway_name(pathway, df.index)
        if r is None:
            print(f"[WARNING] '{pathway}' not found in '{cl} / {key}' "
                  f"- treated as {'0 (penalized)' if penalize_missing else 'missing (excluded)'}")
            vals.append(0.0 if penalize_missing else None)
            continue
        row = df.loc[r]
        if isinstance(row, pd.DataFrame):
            row = row.iloc[0]
        vals.append(row['-log(p-value)'])

    if not penalize_missing:
        vals = [v for v in vals if v is not None]

    return np.mean(vals) if vals else -np.inf


def filter_pathway_groups_top_n(pathway_groups, ipa_results, comparisons,
                                  n_per_group=3, penalize_missing=True):
    """
    Reduce each group in pathway_groups to its top `n_per_group` pathways,
    ranked by mean -log10(p-value) across all comparisons. Missing clones
    are penalized as -log10(1)=0 by default (see _mean_neglogp), so ranking
    rewards recurrence across clones, not just peak significance in one.
    Groups with fewer than n_per_group pathways are returned unchanged.
    """
    filtered = OrderedDict()
    for group_name, pathways in pathway_groups.items():
        scored = [
            (pw, _mean_neglogp(pw, ipa_results, comparisons, penalize_missing))
            for pw in pathways
        ]
        scored.sort(key=lambda x: x[1], reverse=True)
        filtered[group_name] = [pw for pw, _ in scored[:n_per_group]]
    return filtered

In [ ]:
def pathway_level_dotplot(
    ipa_results,
    pathway_groups,
    comparisons,
    title='Pathway z-score',
    size_cap_p=1e-5,
    max_dot=200,
    z_range=None,
    figsize=None,
    col_spacing=0.85,
    row_height=0.32,
    show_group_labels=True,
    show_group_separators=True,
    group_label_fontsize=None,
    group_gap=0.6,
    right_panel_w=1.3,
    col_colors=None,
    cl_labels=None,
    sort_by_significance=False,
    top_n=None,
    filter_top_n_per_group=False,
    n_per_group=3,
    penalize_missing_in_ranking=True,
    pathway_label_colors=None,
    ylabel=None,
    cl_header_pts=30,
    title_gap_pts=20,
    font_sz=18,
    legend_font_sz=None,
    size_legend_x=0.5,
    save_path=None,
):

    # -- Resolve font sizes relative to font_sz --------------------------------
    # Any size that was previously a smaller fixed default now scales off
    # font_sz, preserving the same relative offset it used to have.
    if legend_font_sz is None:
        legend_font_sz = font_sz - 1        # was 8 vs font_sz=9
    if group_label_fontsize is None:
        group_label_fontsize = font_sz - 2  # was 7 vs font_sz=9
    xtick_labelsize = font_sz - 1           # was hardcoded 8 vs font_sz=9

    # -- Fuzzy pathway-name resolver (per comparison's own index) -------------
    def _resolve(name, index):
        if name in index:
            return name
        low = index.str.lower()
        exact = index[low == name.lower()]
        if len(exact):
            return exact[0]
        sub = index[low.str.contains(name.lower(), regex=False)]
        return sub[0] if len(sub) else None

    # -- Optional: reduce each group to its top-N most significant pathways -----
    if filter_top_n_per_group and isinstance(pathway_groups, dict):
        pathway_groups = filter_pathway_groups_top_n(
            pathway_groups, ipa_results, comparisons,
            n_per_group=n_per_group,
            penalize_missing=penalize_missing_in_ranking,
        )

    # -- Build flat pathway list, y-positions (with group padding), separators,
    #    and group-label positions ----------------------------------------------
    # Each group gets `group_gap` rows of empty space above it (including the
    # very first group), and the group label / separator line are centered in
    # that empty space instead of sitting flush against the row above them.
    # With group_gap=0 this reduces exactly to the old tightly-packed layout.
    pathway_to_group = {}
    if isinstance(pathway_groups, dict):
        pathway_order = []
        y_positions   = []   # y-coordinate for each pathway in pathway_order
        sep_indices   = []   # y-position of the separator line between groups
        group_mids    = []   # (group_name, y-position for that group's label)
        y = 0.0
        for gi, (group_name, pws) in enumerate(pathway_groups.items()):
            y += group_gap                        # padding above this group
            label_y = y - group_gap / 2 - 0.5      # centered in that padding
            group_mids.append((group_name, label_y))
            if gi > 0:
                sep_indices.append(label_y)        # no separator above the first group
            for p in pws:
                pathway_order.append(p)
                y_positions.append(y)
                pathway_to_group[p] = group_name
                y += 1
    else:
        pathway_order = list(pathway_groups)
        y_positions   = list(range(len(pathway_order)))
        sep_indices   = []
        group_mids    = []

    n_paths = len(pathway_order)
    n_comps = len(comparisons)

    # -- Extract z-score and -log(p) matrices ----------------------------------
    z_mat  = np.full((n_paths, n_comps), np.nan)
    lp_mat = np.full((n_paths, n_comps), np.nan)
    resolved_names = list(pathway_order)

    for j, (cl, key, _) in enumerate(comparisons):
        # results_all_ipa is a flat dict keyed like "1806_Fusion C2C4 vs MPV"
        # so match on cell line + clone-pair substrings rather than nested lookup
        candidates = [k for k in ipa_results if cl in k and key in k]
        if not candidates:
            print(f"[WARNING] no IPA table found for cl={cl}, key={key}")
            continue
        if len(candidates) > 1:
            print(f"[WARNING] multiple IPA table matches for cl={cl}, key={key}: {candidates} - using first")
        ipa = ipa_results[candidates[0]]
        df = ipa.pathways

        idx = df.index
        for i, pw in enumerate(pathway_order):
            r = _resolve(pw, idx)
            if r is None:
                print(f"[WARNING] '{pw}' not found in '{cl} / {key}'")
                continue
            if j == 0:
                resolved_names[i] = r
            row = df.loc[r]
            if isinstance(row, pd.DataFrame):  # guard against duplicate index entries
                row = row.iloc[0]
            z_mat[i, j]  = row['zScore']
            lp_mat[i, j] = row['-log(p-value)']

    display_order = pathway_order  # y-axis labels track requested names, not resolved ones

    # -- Sort by significance and apply top_n -----------------------------------
    if sort_by_significance:
        with np.errstate(invalid='ignore'):
            max_lp_per_pathway = np.nanmax(lp_mat, axis=1)
        all_nan_mask = np.all(np.isnan(lp_mat), axis=1)
        max_lp_per_pathway = np.where(all_nan_mask, -np.inf, max_lp_per_pathway)
        sort_idx = np.argsort(-max_lp_per_pathway, kind='stable')  # descending significance
        if top_n is not None:
            sort_idx = sort_idx[:top_n]
        display_order  = [display_order[i] for i in sort_idx]
        resolved_names = [resolved_names[i] for i in sort_idx]
        z_mat   = z_mat[sort_idx]
        lp_mat  = lp_mat[sort_idx]
        n_paths = len(display_order)
        sep_indices = []
        group_mids  = []
        y_positions = list(range(n_paths))  # group structure discarded, so no more gaps

    # -- Dot sizes ---------------------------------------------------------------
    fixed_max_lp = -np.log10(size_cap_p)
    lp_mat_clean = np.where(np.isnan(lp_mat), 0, lp_mat)
    lp_mat_clean = np.where(np.isinf(lp_mat_clean), fixed_max_lp, lp_mat_clean)
    size_mat = np.clip((lp_mat_clean / fixed_max_lp) * max_dot, 2, max_dot)

    # -- Color scale ---------------------------------------------------------------
    if z_range is None:
        valid = z_mat[~np.isnan(z_mat)]
        z_range = float(np.max(np.abs(valid))) if len(valid) else 1.0
        z_range = max(z_range, 0.5)

    norm = TwoSlopeNorm(vmin=-z_range, vcenter=0, vmax=z_range)
    cmap = colormaps['coolwarm']

    # -- Vertical cell-line separator positions -----------------------------------
    cl_seps = []
    for j in range(n_comps - 1):
        if comparisons[j][0] != comparisons[j + 1][0]:
            cl_seps.append(j + 0.5)

    # -- Figure ---------------------------------------------------------------------
    # y_span accounts for the extra room group_gap adds; with group_gap=0 this is
    # identical to n_paths, matching the old figure-height calculation exactly.
    y_span = (max(y_positions) - min(y_positions) + 1) if y_positions else 1
    main_w = n_comps * col_spacing + 3.2
    if figsize is None:
        figsize = (main_w + right_panel_w, y_span * row_height + 2.5)

    fig = plt.figure(figsize=figsize)
    gs  = fig.add_gridspec(
        2, 2,
        width_ratios  = [main_w, right_panel_w],
        height_ratios = [3, 2],
        wspace        = 0.08,
        hspace        = 0.40,
    )
    ax  = fig.add_subplot(gs[:, 0])
    cax = fig.add_subplot(gs[0, 1])
    lax = fig.add_subplot(gs[1, 1])
    lax.axis('off')

    def _pt_up(pts):
        return mtransforms.ScaledTranslation(0, pts / 72, fig.dpi_scale_trans)

    # -- Scatter dots -----------------------------------------------------------------
    for i in range(n_paths):
        for j in range(n_comps):
            color = 'lightgrey' if np.isnan(z_mat[i, j]) else cmap(norm(z_mat[i, j]))
            ax.scatter(j, y_positions[i],
                       s=size_mat[i, j],
                       color=color,
                       edgecolors='none',
                       zorder=3)

    # -- Horizontal group separators (independent toggle from group labels) -------
    if show_group_separators:
        for sep in sep_indices:
            ax.axhline(sep, color='black', linewidth=0.8, linestyle='--', zorder=4)

    for sep in cl_seps:
        ax.axvline(sep, color='black', linewidth=0.8, linestyle='-', zorder=4)

    # -- Y-axis label colouring -------------------------------------------------------
    use_label_colors = (
        show_group_labels
        and pathway_label_colors is not None
        and isinstance(pathway_groups, dict)
    )

    ax.set_yticks(y_positions)
    ax.set_yticklabels(display_order, fontsize=font_sz)

    if use_label_colors:
        for tick_label, pw in zip(ax.get_yticklabels(), display_order):
            grp   = pathway_to_group.get(pw)
            color = pathway_label_colors.get(grp, 'black')
            tick_label.set_color(color)

    if show_group_labels and group_mids and not use_label_colors:
        blend = mtransforms.blended_transform_factory(ax.transAxes, ax.transData)
        for group_name, label_y in group_mids:
            ax.text(-0.01, label_y, f"- {group_name}",
                    transform=blend,
                    ha='right', va='center',
                    fontsize=group_label_fontsize,
                    color='#888888',
                    clip_on=False)

    if ylabel is not None:
        ax.set_ylabel(ylabel, fontsize=font_sz, labelpad=8, fontweight='bold')

    # -- Axis formatting ----------------------------------------------------------------
    ax.set_xticks(range(n_comps))
    ax.set_xticklabels([])
    ax.set_xlim(-0.6, n_comps - 0.4)
    y_min = min(y_positions) if y_positions else 0
    y_max = max(y_positions) if y_positions else 0
    ax.set_ylim(y_min - 0.6, y_max + 0.6)
    ax.invert_yaxis()
    ax.tick_params(top=True, labeltop=False, bottom=False, labelbottom=False)
    ax.tick_params(axis='x', labelsize=xtick_labelsize)

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('black')
        spine.set_linewidth(0.8)

    blend_top = mtransforms.blended_transform_factory(ax.transData, ax.transAxes)

    # -- Condition sub-labels -------------------------------------------------------------
    for j, (cl, key, label) in enumerate(comparisons):
        color = (col_colors or {}).get(key, 'black')
        ax.text(j, 1.0, label,
                transform=blend_top + _pt_up(5),
                ha='center', va='bottom',
                fontsize=font_sz, color=color,
                clip_on=False)

    # -- Cell-line group headers -----------------------------------------------------------
    if cl_labels:
        cl_runs = [
            (cl, [j for j, _ in group])
            for cl, group in groupby(enumerate(comparisons), key=lambda x: x[1][0])
        ]
        for cl, indices in cl_runs:
            x_mid   = (indices[0] + indices[-1]) / 2
            x_l     = indices[0]  - 0.35
            x_r     = indices[-1] + 0.35
            display = cl_labels.get(cl, cl)

            ax.text(x_mid, 1.0, display,
                    transform=blend_top + _pt_up(cl_header_pts),
                    ha='center', va='bottom',
                    fontsize=font_sz, fontweight='bold',
                    clip_on=False)

            ax.plot([x_l, x_r], [1.0, 1.0],
                    transform=blend_top + _pt_up(cl_header_pts - 5),
                    color='black', lw=0.8,
                    clip_on=False)

    # -- Title -------------------------------------------------------------------------------
    title_pts = (cl_header_pts + title_gap_pts) if cl_labels else 10
    ax.text(0.5, 1.0, title,
            transform=ax.transAxes + _pt_up(title_pts),
            ha='center', va='bottom',
            fontsize=font_sz, fontweight='bold',
            clip_on=False)

    # -- Colorbar --------------------------------------------------------------------------
    sm = ScalarMappable(norm=norm, cmap='coolwarm')
    sm.set_array([])
    fig.colorbar(sm, cax=cax)
    cax.set_ylabel('Z-score', fontsize=legend_font_sz)
    cax.tick_params(labelsize=legend_font_sz)

    # -- Size legend --------------------------------------------------------------------------
    p_levels    = sorted(set([0.1, 1e-2, 1e-3, 1e-4, size_cap_p]))
    leg_handles = []
    for p in p_levels:
        lp  = -np.log10(p)
        s   = np.clip((lp / fixed_max_lp) * max_dot, 2, max_dot)
        ms  = 2 * np.sqrt(s / np.pi)
        leg_handles.append(
            Line2D([0], [0], marker='o', color='w',
                   markerfacecolor='grey', markeredgecolor='none',
                   markersize=ms, label=f'p = {p:.0e}')
        )

    if use_label_colors:
        shown_groups = {
            pathway_to_group[p] for p in display_order
            if pathway_to_group.get(p) in pathway_label_colors
        }
        color_handles = [
            Line2D([0], [0], marker='s', color='w',
                   markerfacecolor=pathway_label_colors[grp], markeredgecolor='none',
                   markersize=8, label=grp)
            for grp in pathway_label_colors
            if grp in shown_groups
        ]
        color_leg = lax.legend(
            handles=color_handles,
            title='Pathway group', title_fontsize=font_sz,
            loc='upper center',
            frameon=False, fontsize=font_sz,
            handletextpad=0.5, labelspacing=0.8,
        )
        lax.add_artist(color_leg)
        size_leg_loc = 'lower center'
    else:
        size_leg_loc = 'upper center'

    # bbox_to_anchor lets us slide the legend horizontally within lax:
    # size_legend_x=0.5 is centered (old behavior); push it toward 1.0+ to
    # move it right, e.g. 0.5 -> mostly-right -> fully outside the panel.
    size_leg_anchor_y = 1.0 if size_leg_loc == 'upper center' else 0.0
    lax.legend(handles=leg_handles,
               title='Adjusted\np-value', title_fontsize=legend_font_sz,
               loc=size_leg_loc,
               bbox_to_anchor=(size_legend_x, size_leg_anchor_y),
               bbox_transform=lax.transAxes,
               frameon=False, fontsize=legend_font_sz,
               handletextpad=1.0, labelspacing=1.2)

    # -- Layout ------------------------------------------------------------------------------
    plt.tight_layout()

    if show_group_labels and group_mids and not use_label_colors:
        fig.subplots_adjust(left=fig.subplotpars.left + 0.12)

    top_margin_in = (title_pts + 14) / 72
    fig.subplots_adjust(
        top=fig.subplotpars.top - top_margin_in / figsize[1]
    )

    pos = cax.get_position()
    cax.set_position([pos.x0, pos.y0, pos.width * 0.25, pos.height])

    if save_path:
        plt.savefig(save_path, bbox_inches='tight')

    plt.show()
    return fig, ax

In [ ]:
def pathway_level_table(
    ipa_results,
    pathway_groups,
    comparisons,
    sort_by_significance=False,
    top_n=None,
    filter_top_n_per_group=False,
    n_per_group=3,
    penalize_missing_in_ranking=True,
    save_path=None,
):
    """
    Companion to pathway_level_dotplot: produces a table of the same pathways,
    in the same order, with zScore and -log10(p-value) for each comparison.

    Mirrors pathway_level_dotplot's fuzzy name resolution (via _resolve_pathway_name),
    optional top-N-per-group filtering (via filter_pathway_groups_top_n), and
    significance-based sorting (by each pathway's max -log10(p-value) across
    comparisons, descending).

    Returns a DataFrame with columns:
        pathway, <label>_zScore, <label>_negLogP, <label>_pvalue,
                 <label>_zScore, <label>_negLogP, <label>_pvalue, ...
    one triplet per entry in `comparisons`, in the order given.
    """

    # -- Optional: reduce each group to its top-N most significant pathways ---
    # (identical to pathway_level_dotplot)
    if filter_top_n_per_group and isinstance(pathway_groups, dict):
        pathway_groups = filter_pathway_groups_top_n(
            pathway_groups, ipa_results, comparisons,
            n_per_group=n_per_group,
            penalize_missing=penalize_missing_in_ranking,
        )

    # -- Build flat pathway list (mirrors pathway_level_dotplot) --------------
    if isinstance(pathway_groups, dict):
        pathway_order = []
        for pws in pathway_groups.values():
            pathway_order.extend(pws)
    else:
        pathway_order = list(pathway_groups)

    n_paths = len(pathway_order)
    n_comps = len(comparisons)

    # -- Extract z-score and -log(p) matrices (mirrors pathway_level_dotplot) -
    z_mat  = np.full((n_paths, n_comps), np.nan)
    lp_mat = np.full((n_paths, n_comps), np.nan)

    for j, (cl, key, _) in enumerate(comparisons):
        candidates = [k for k in ipa_results if cl in k and key in k]
        if not candidates:
            print(f"[WARNING] no IPA table found for cl={cl}, key={key}")
            continue
        if len(candidates) > 1:
            print(f"[WARNING] multiple IPA table matches for cl={cl}, key={key}: {candidates} - using first")
        ipa = ipa_results[candidates[0]]
        df = ipa.pathways
        idx = df.index

        for i, pw in enumerate(pathway_order):
            r = _resolve_pathway_name(pw, idx)
            if r is None:
                print(f"[WARNING] '{pw}' not found in '{cl} / {key}'")
                continue
            row = df.loc[r]
            if isinstance(row, pd.DataFrame):  # guard against duplicate index entries
                row = row.iloc[0]
            z_mat[i, j]  = row['zScore']
            lp_mat[i, j] = row['-log(p-value)']

    # -- Sort by significance and apply top_n (mirrors pathway_level_dotplot) -
    if sort_by_significance:
        with np.errstate(invalid='ignore'):
            max_lp_per_pathway = np.nanmax(lp_mat, axis=1)
        all_nan_mask = np.all(np.isnan(lp_mat), axis=1)
        max_lp_per_pathway = np.where(all_nan_mask, -np.inf, max_lp_per_pathway)
        sort_idx = np.argsort(-max_lp_per_pathway, kind='stable')  # descending significance
        if top_n is not None:
            sort_idx = sort_idx[:top_n]
        pathway_order = [pathway_order[i] for i in sort_idx]
        z_mat   = z_mat[sort_idx]
        lp_mat  = lp_mat[sort_idx]
        n_paths = len(pathway_order)

    # -- Assemble output dataframe ---------------------------------------------
    out = pd.DataFrame({'pathway': pathway_order})
    for j, (cl, key, label) in enumerate(comparisons):
        out[f'{label}_zScore']  = z_mat[:, j]
        out[f'{label}_negLogP'] = lp_mat[:, j]
        with np.errstate(over='ignore'):
            out[f'{label}_pvalue'] = 10.0 ** (-lp_mat[:, j])

    if save_path:
        out.to_csv(save_path, sep='\t' if save_path.endswith('.tsv') else ',', index=False)

    return out

In [ ]:
############# exploratory pathway gene group creation ##############################

### first merge IPA results into a dictionary so genes can be found in both easily
results_all_ipa = (
    {f"1806_{k}": v for k, v in results_1806.items()} |
    {f"231_{k}":  v for k, v in results_231.items()}
)

def build_gene_groups(
    ipa_dict,
    results,
    comparisons,
    pathway_groups,          # OrderedDict: {display_name: [pathway, ...], ...}
    min_padj    = 0.1,
    min_lfc     = 0.3,
    remove_overlap = True,   # remove genes already assigned to earlier groups
):
    """
    Build a gene_groups dict ready for gene_level_dotplot.

    Parameters
    ----------
    ipa_dict        : merged IPA results dict (e.g. results_all_ipa)
    results         : DESeq results dict keyed by (cell_line, comparison)
    comparisons     : list of (cell_line, comp, label) tuples
    pathway_groups  : OrderedDict mapping display name -> list of IPA pathway names
    min_padj        : adjusted p-value cutoff for signal filter
    min_lfc         : minimum |log2FC| for signal filter
    remove_overlap  : if True, genes assigned to an earlier group are removed
                      from all later groups (first-group priority)

    Returns
    -------
    gene_groups : dict mapping display name -> filtered gene list
    """

    def _extract(pathway_names):
        genes = set()
        for label, ipa in ipa_dict.items():
            for pathway in pathway_names:
                if pathway in ipa.pathways.index:
                    row = ipa.pathways.loc[pathway]
                    new_genes = [g.strip() for g in row['Molecules'].split(',')]
                    genes.update(new_genes)
        return sorted(genes)

    def _filter(gene_list):
        keep = []
        for gene in gene_list:
            for cell_line, comp, label in comparisons:
                df = results[cell_line][comp]
                if gene in df.index:
                    row = df.loc[gene]
                    if (pd.notna(row['f_mpv_padj'])
                            and row['f_mpv_padj'] < min_padj
                            and abs(row['f_mpv_log2FoldChange']) >= min_lfc):
                        keep.append(gene)
                        break
        return keep

    gene_groups  = {}
    assigned     = set()   # tracks genes already placed in an earlier group

    for display_name, pathway_names in pathway_groups.items():
        # 1. extract
        genes = _extract(pathway_names)

        # 2. remove overlap with previously assigned groups
        if remove_overlap:
            genes = [g for g in genes if g not in assigned]

        # 3. filter by signal
        genes = _filter(genes)

        gene_groups[display_name] = genes
        assigned.update(genes)

        print(f"{display_name}: {len(genes)} genes - {genes}")

    return gene_groups

### All key groups

In [ ]:
comparisons = [
    ("1806", "C2C4", "C2C4"),
    ("1806", "C5C2", "C5C2"),
    ("1806", "C6C7", "C6C7"),
    ("231",  "C1C4", "C1C4"),
    ("231",  "C6C8", "C6C8"),
]

pathway_groups = OrderedDict({
    'Translation / RiBi': [
        'Major pathway of rRNA processing in the nucleolus and cytosol',
        'Eukaryotic Translation Initiation',
        'Eukaryotic Translation Elongation',
        'Eukaryotic Translation Termination',
        'SRP-dependent cotranslational protein targeting to membrane',
        'Nonsense-Mediated Decay (NMD)',
        'Ribosomal Quality Control Signaling Pathway',
        'Response of EIF2AK4 (GCN2) to amino acid deficiency',
        'rRNA modification in the nucleus and cytosol',
        'Regulation of eIF4 and p70S6K Signaling',
        'Selenoamino acid metabolism',
        'Exosome Signaling Pathway',
        'Regulation of mRNA stability by proteins that bind AU-rich elements',
    ],
    'Cell Cycle': [
        'Mitotic G2-G2/M phases',
        'Mitotic G1 phase and G1/S transition',
        'S Phase',
        'Mitotic Metaphase and Anaphase',
        'Cell Cycle Checkpoints',
        'Regulation of mitotic cell cycle',
        'DNA Replication Pre-Initiation',
        'Synthesis of DNA',
        'Cell Cycle: G2/M DNA Damage Checkpoint Regulation',
    ],
    'Proteostasis / Protein Turnover': [
        'Proteasome assembly',
        'Protein Ubiquitination Pathway',
        'Neddylation',
        'NIK-->noncanonical NF-kB signaling',
        'Deubiquitination',
        'Degradation of beta-catenin by the destruction complex',
        'Degradation of CRY and PER proteins',
        'TNFR2 non-canonical NF-kB pathway',
        'HSP90 chaperone cycle for steroid hormone receptor',
        'Protein folding',
        'Post-translational protein phosphorylation',
    ],
    'Mitochondria / OXPHOS': [
        'Oxidative Phosphorylation',
        'Respiratory electron transport',
        'Cristae formation',
        'Mitochondrial translation',
        'Mitochondrial protein degradation',
        'tRNA processing in the mitochondrion',
        'Complex IV assembly',
        'Mitochondrial protein import',
    ],
    'DNA Damage Response & Repair': [
        'Nucleotide Excision Repair',
        'NER (Nucleotide Excision Repair, Enhanced Pathway)',
        'Resolution of Abasic Sites (AP sites)',
        'BER (Base Excision Repair) Pathway',
        'HDR through MMEJ (alt-NHEJ)',
        'Role of CHK Proteins in Cell Cycle Checkpoint Control',
        'DNA damage-induced 14-3-3σ Signaling',
        'DNA Damage/Telomere Stress Induced Senescence',
    ],
    'RNA Processing / Modification': [
        'RNA m6A Methylation Signaling Pathway',
        'Metabolism of non-coding RNA',
        'Processing of Capped Intron-Containing Pre-mRNA',
        'RNA Polymerase II Transcription',
        'mRNA Capping',
        'RNA polymerase II transcribes snRNA genes',
        'Maternal to Zygotic Transition Signaling Pathway',
        'XIST Epigenetic Regulation Signaling Pathway',
    ],
    'Hedgehog / Developmental Signaling': [
        'Hedgehog ligand biogenesis',
        'Hedgehog \'off\' state',
        'Hedgehog \'on\' state',
        'Signaling by NOTCH4',
        'TCF dependent signaling in response to WNT',
        'Transcriptional activity of SMAD2/SMAD3:SMAD4 heterotrimer',
        'Signaling by TGF-beta Receptor Complex',
        'Cardiac Valve Formation Signaling Pathway',
        'Cerebral Malformation Signaling Pathway',
        'Somitogenesis',
        'Mouse Embryonic Stem Cell Pluripotency',
    ],
    'Cytoskeleton / Rho-GTPase / MAPK Signaling': [
        'RHO GTPase cycle',
        'RHOGDI Signaling',
        'Signaling by Rho Family GTPases',
        'RHO GTPases Activate Formins',
        'RHO GTPases activate IQGAPs',
        'RAC Signaling',
        'Actin Cytoskeleton Signaling',
        'Actin Nucleation by ARP-WASP Complex',
        'Paxillin Signaling',
        'ILK Signaling',
        'ERK/MAPK Signaling',
        'RAF/MAP kinase cascade',
        'RAF-independent MAPK1/3 activation',
        'MAPK6/MAPK4 signaling',
        'Ephrin Receptor Signaling',
        'Integrin to Cytoskeleton Signaling Pathway',
        'Cell junction organization',
        'Clathrin-mediated endocytosis',
        'Regulation of Cellular Mechanics by Calpain Protease',
        'Neuron Navigator Signaling Pathway',
    ],
    'Interferon / Antiviral Immune': [
        'Interferon alpha/beta signaling',
        'Interferon gamma signaling',
        'Antigen Presentation Pathway',
        'Modulation of host responses by IFN-stimulated genes',
        'Role of PKR in Interferon Induction and Antiviral Response',
        'Role of RIG1-like Receptors in Antiviral Innate Immunity',
        'Cytosolic sensors of pathogen-associated DNA',
        'GAIT Translation Signaling Pathway',
    ],
    'Cytokine / Inflammatory Signaling': [
        'IL-8 Signaling',
        'Chemokine Signaling',
        'Other interleukin signaling',
        'Interleukin-1 family signaling',
        'Role of IL-17F in Allergic Inflammatory Airway Diseases',
        'TNFR1 Signaling',
        'TNFR2 Signaling',
        'TCR signaling',
        'Signaling by the B Cell Receptor (BCR)',
        'Fc epsilon receptor (FCERI) signaling',
        'B Cell Activating Factor Signaling',
        'C-type lectin receptors (CLRs)',
        'Neutrophil degranulation',
        'Oncostatin M Signaling',
        'fMLP Signaling in Neutrophils',
        'Interleukin-10 signaling',
        'Role of Macrophages, Fibroblasts and Endothelial Cells in Rheumatoid Arthritis',
        'Acute Phase Response Signaling',
        'Tumor Microenvironment Pathway',
        'Hematoma Resolution Signaling Pathway',
        'Gene and protein expression by JAK-STAT signaling',
    ],
    'ECM / Fibrosis / Collagen': [
        'Assembly of collagen fibrils and other multimeric structures',
        'Collagen biosynthesis and modifying enzymes',
        'Extracellular matrix organization',
        'Hepatic Fibrosis Signaling Pathway',
        'Pulmonary Fibrosis Idiopathic Signaling Pathway',
        'Formation of the dystrophin-glycoprotein complex',
        'Keratinization',
    ],
    'Cellular Stress Response': [
        'Cellular response to heat stress',
        'Cellular response to hypoxia',
        'XBP1(S) activates chaperone genes',
        'NFE2L2 regulating anti-oxidant/detoxification enzymes',
        'KEAP1-NFE2L2 pathway',
        'Cachexia Signaling Pathway',
        'Response of EIF2AK1 (HRI) to heme deficiency',
    ],
    'Hormone / Nuclear Receptor Signaling': [
        'ESR-mediated signaling',
        'Extra-nuclear estrogen signaling',
        'TR/RXR Activation',
        'VDR/RXR Activation',
        'Oxytocin Signaling Pathway',
        'Oxytocin in Brain Signaling Pathway',
        'α-Adrenergic Signaling',
    ],
    'Growth Factor / RTK Signaling': [
        'IGF-1 Signaling',
        'Regulation of Insulin-like Growth Factor (IGF) transport and uptake by IGFBPs',
        'NGF Signaling',
        'MSP-RON Signaling in Cancer Cells Pathway',
        'Signaling by MET',
        'GP6 Signaling Pathway',
        'Thrombin Signaling',
        'Signaling by VEGF',
    ],
    'Other': [
        'ABC-family proteins mediated transport',
        'ABRA Signaling Pathway',
        'Apoptotic execution phase',
        'Centrosomal KIAA0586 Signaling Pathway',
        'Cilia Biogenesis Signaling Pathway',
        'DYRK1A Signaling Pathway',
        'Epithelial Membrane Protein Signaling Pathway',
        'Folate Signaling Pathway',
        'G-Protein Coupled Receptor Signaling',
        'Glycation Signaling Pathway',
        'Hereditary Breast Cancer Signaling',
        'Interconversion of nucleotide di- and triphosphates',
        'L1CAM interactions',
        'Metabolism of polyamines',
        'Microautophagy Signaling Pathway',
        'Myelination Signaling Pathway',
        'Parkinson\'s Signaling Pathway',
        'Plasma lipoprotein assembly, remodeling, and clearance',
        'Regulation of Apoptosis',
        'Regulation of RUNX2 expression and activity',
        'Role of Tissue Factor in Cancer',
        'Sertoli Cell-Germ Cell Junction Signaling Pathway',
        'Sertoli Cell-Sertoli Cell Junction Signaling',
        'Signaling by Hippo',
        'Signaling by ROBO receptors',
        'Transcriptional regulation by RUNX1',
        'Transcriptional regulation by RUNX3',
        'Type II Diabetes Mellitus Signaling',
    ],
})

fig, ax = pathway_level_dotplot(
    ipa_results = results_all_ipa,
    pathway_groups = pathway_groups,
    comparisons = comparisons,
    cl_labels   = {"1806": "HCC1806", "231": "MDA-MB-231"},
    title       = 'Fusion Clonal Populations Top IPA Pathways Grouped',
    z_range     = 5,
    size_cap_p  = 1e-5,
    show_group_labels=True,
    show_group_separators=True,
    filter_top_n_per_group=True,
    n_per_group=3,
    penalize_missing_in_ranking=True,
    save_path   = 'pathway_dotplot.svg',
    font_sz=18,
    col_spacing=0.5,
    row_height=0.35,
    max_dot=400,
    size_legend_x=0.9,
)

pathway_table = pathway_level_table(
    ipa_results = results_all_ipa,
    pathway_groups = pathway_groups,
    comparisons = comparisons,
    sort_by_significance=True,
    top_n=None,
    filter_top_n_per_group=True,
    n_per_group=3,
    penalize_missing_in_ranking=True,
    save_path='pathway_dotplot_table.tsv',
)
print(pathway_table)

## Gene level plots FINAL

In [ ]:
def _rank_genes(padj_mat, rank_method='min_p'):
    """
    Compute a sort order (best/most-significant first) over genes given a
    (n_genes x n_comparisons) padj matrix.

    rank_method='min_p' (default, matches prior gene_level_dotplot/table behavior):
        Rank by each gene's single lowest padj across all comparisons. A gene
        need only be extreme in ONE comparison to rank highly - this favors
        genes driven by whichever comparisons have the most statistical power,
        regardless of whether the effect recurs elsewhere.

    rank_method='mean_p_penalized' (mirrors _mean_neglogp / filter_pathway_groups_top_n
        at the pathway level):
        Rank by each gene's mean -log10(padj) across ALL comparisons, where
        comparisons in which the gene is absent/non-significant (NaN padj)
        are penalized to -log10(1) = 0 rather than excluded from the average.
        This rewards genes that are recurrently significant across multiple
        comparisons, and down-ranks genes that are only extreme in a single
        comparison - the same recurrence-based logic already used to pick
        top pathways per group.

    Returns an index array (ascending = best first) suitable for use as
    sort_idx on lfc_mat / padj_mat / gene_order.
    """
    if rank_method == 'min_p':
        with np.errstate(invalid='ignore'):
            score = np.nanmin(padj_mat, axis=1)
        all_nan_mask = np.all(np.isnan(padj_mat), axis=1)
        score = np.where(all_nan_mask, np.inf, score)
        sort_idx = np.argsort(score, kind='stable')  # ascending: lowest p first

    elif rank_method == 'mean_p_penalized':
        with np.errstate(divide='ignore', invalid='ignore'):
            neglogp = -np.log10(padj_mat)
        neglogp = np.where(np.isnan(neglogp), 0.0, neglogp)  # penalize missing/non-sig -> 0
        mean_neglogp = np.mean(neglogp, axis=1)
        sort_idx = np.argsort(-mean_neglogp, kind='stable')  # descending: highest mean sig first

    else:
        raise ValueError(
            f"Unknown rank_method: {rank_method!r}. Use 'min_p' or 'mean_p_penalized'."
        )

    return sort_idx

def gene_level_dotplot(
    results,
    gene_groups,
    comparisons,
    title='Gene-level log$_2$FC',
    size_cap_p=1e-5,
    max_dot=200,
    lfc_range=None,
    figsize=None,
    col_spacing=0.85,
    row_height=0.32,
    show_group_labels=True,
    group_label_fontsize=None,
    group_gap=0.6,
    right_panel_w=1.3,
    col_colors=None,
    cl_labels=None,
    sort_by_significance=False,
    rank_method='min_p',   # 'min_p' (default, prior behavior) or 'mean_p_penalized'
    top_n=None,
    gene_label_colors=None,
    ylabel=None,
    cl_header_pts=30,
    title_gap_pts=20,
    font_sz=18,
    legend_font_sz=None,
    size_legend_x=0.5,
    save_path=None,
):

    # -- Resolve font sizes relative to font_sz --------------------------------
    # Any size that was previously a smaller fixed default now scales off
    # font_sz, preserving the same relative offset it used to have.
    if legend_font_sz is None:
        legend_font_sz = font_sz - 1        # was 8 vs font_sz=9
    if group_label_fontsize is None:
        group_label_fontsize = font_sz - 2  # was 7 vs font_sz=9
    xtick_labelsize = font_sz - 1           # was hardcoded 8 vs font_sz=9

    # -- Build flat gene list, y-positions (with group padding), separators,
    #    and group-label positions ----------------------------------------------
    # Each group gets `group_gap` rows of empty space above it (including the
    # very first group), and the group label / separator line are centered in
    # that empty space instead of sitting flush against the row above them.
    # With group_gap=0 this reduces exactly to the old tightly-packed layout.
    gene_to_group = {}
    if isinstance(gene_groups, dict):
        gene_order  = []
        y_positions = []   # y-coordinate for each gene in gene_order
        sep_indices = []   # y-position of the separator line between groups
        group_mids  = []   # (group_name, y-position for that group's label)
        y = 0.0
        for gi, (group_name, genes) in enumerate(gene_groups.items()):
            y += group_gap                        # padding above this group
            label_y = y - group_gap / 2 - 0.5      # centered in that padding
            group_mids.append((group_name, label_y))
            if gi > 0:
                sep_indices.append(label_y)        # no separator above the first group
            for g in genes:
                gene_order.append(g)
                y_positions.append(y)
                gene_to_group[g] = group_name
                y += 1
    else:
        gene_order  = list(gene_groups)
        y_positions = list(range(len(gene_order)))
        sep_indices = []
        group_mids  = []

    n_genes = len(gene_order)
    n_comps = len(comparisons)

    # -- Extract LFC and padj matrices ----------------------------------------
    lfc_mat  = np.full((n_genes, n_comps), np.nan)
    padj_mat = np.full((n_genes, n_comps), np.nan)

    for j, (cl, key, _) in enumerate(comparisons):
        df = results.get(cl, {}).get(key, pd.DataFrame())
        if df.empty:
            continue
        for i, gene in enumerate(gene_order):
            if gene in df.index:
                lfc_mat[i, j]  = df.loc[gene, 'f_mpv_log2FoldChange']
                padj_mat[i, j] = df.loc[gene, 'f_mpv_padj']

    # -- Sort by significance and apply top_n ----------------------------------
    if sort_by_significance:
        sort_idx = _rank_genes(padj_mat, rank_method=rank_method)
        if top_n is not None:
            sort_idx = sort_idx[:top_n]
        gene_order  = [gene_order[i] for i in sort_idx]
        lfc_mat     = lfc_mat[sort_idx]
        padj_mat    = padj_mat[sort_idx]
        n_genes     = len(gene_order)
        sep_indices = []
        group_mids  = []
        y_positions = list(range(n_genes))  # group structure discarded, so no more gaps

    # -- Dot sizes -------------------------------------------------------------
    fixed_max_lp = -np.log10(size_cap_p)

    with np.errstate(divide='ignore'):
        lp_mat = -np.log10(padj_mat)

    lp_mat   = np.where(np.isnan(lp_mat), 0, lp_mat)
    lp_mat   = np.where(np.isinf(lp_mat), fixed_max_lp, lp_mat)
    size_mat = np.clip((lp_mat / fixed_max_lp) * max_dot, 2, max_dot)

    # -- Color scale -----------------------------------------------------------
    if lfc_range is None:
        valid = lfc_mat[~np.isnan(lfc_mat)]
        lfc_range = float(np.max(np.abs(valid))) if len(valid) else 1.0
        lfc_range = max(lfc_range, 0.5)

    norm = TwoSlopeNorm(vmin=-lfc_range, vcenter=0, vmax=lfc_range)
    cmap = colormaps['coolwarm']

    # -- Vertical cell-line separator positions --------------------------------
    cl_seps = []
    for j in range(n_comps - 1):
        if comparisons[j][0] != comparisons[j + 1][0]:
            cl_seps.append(j + 0.5)

    # -- Figure ----------------------------------------------------------------
    # y_span accounts for the extra room group_gap adds; with group_gap=0 this is
    # identical to n_genes, matching the old figure-height calculation exactly.
    y_span = (max(y_positions) - min(y_positions) + 1) if y_positions else 1
    main_w = n_comps * col_spacing + 3.2
    if figsize is None:
        figsize = (main_w + right_panel_w, y_span * row_height + 2.5)

    fig = plt.figure(figsize=figsize)
    gs  = fig.add_gridspec(
        2, 2,
        width_ratios  = [main_w, right_panel_w],
        height_ratios = [3, 2],
        wspace        = 0.08,
        hspace        = 0.40,
    )
    ax  = fig.add_subplot(gs[:, 0])
    cax = fig.add_subplot(gs[0, 1])
    lax = fig.add_subplot(gs[1, 1])
    lax.axis('off')

    # -- Helper: fixed-point vertical offset from the top spine ---------------
    def _pt_up(pts):
        return mtransforms.ScaledTranslation(0, pts / 72, fig.dpi_scale_trans)

    # -- Scatter dots ----------------------------------------------------------
    for i in range(n_genes):
        for j in range(n_comps):
            color = 'lightgrey' if np.isnan(lfc_mat[i, j]) else cmap(norm(lfc_mat[i, j]))
            ax.scatter(j, y_positions[i],
                       s=size_mat[i, j],
                       color=color,
                       edgecolors='none',
                       zorder=3)

    # -- Horizontal group separators ------------------------------------------
    for sep in sep_indices:
        ax.axhline(sep, color='black', linewidth=0.8, linestyle='--', zorder=4)

    # -- Vertical cell-line separators ----------------------------------------
    for sep in cl_seps:
        ax.axvline(sep, color='black', linewidth=0.8, linestyle='-', zorder=4)

    # -- Y-axis label colouring -----------------------------------------------
    use_label_colors = (
        show_group_labels
        and gene_label_colors is not None
        and isinstance(gene_groups, dict)
    )

    ax.set_yticks(y_positions)
    ax.set_yticklabels(gene_order, fontsize=font_sz)

    if use_label_colors:
        for tick_label, gene in zip(ax.get_yticklabels(), gene_order):
            grp   = gene_to_group.get(gene)
            color = gene_label_colors.get(grp, 'black')
            tick_label.set_color(color)

    # -- Bracket-style group labels -------------------------------------------
    if show_group_labels and group_mids and not use_label_colors:
        blend = mtransforms.blended_transform_factory(ax.transAxes, ax.transData)
        for group_name, label_y in group_mids:
            ax.text(-0.01, label_y, f"- {group_name}",
                    transform=blend,
                    ha='right', va='center',
                    fontsize=group_label_fontsize,
                    color='#888888',
                    clip_on=False)

    # -- Optional y-axis label ------------------------------------------------
    if ylabel is not None:
        ax.set_ylabel(ylabel, fontsize=font_sz, labelpad=8, fontweight='bold')

    # -- Axis formatting ------------------------------------------------------
    ax.set_xticks(range(n_comps))
    ax.set_xticklabels([])
    ax.set_xlim(-0.6, n_comps - 0.4)
    y_min = min(y_positions) if y_positions else 0
    y_max = max(y_positions) if y_positions else 0
    ax.set_ylim(y_min - 0.6, y_max + 0.6)
    ax.invert_yaxis()
    ax.tick_params(top=True, labeltop=False, bottom=False, labelbottom=False)
    ax.tick_params(axis='x', labelsize=xtick_labelsize)

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('black')
        spine.set_linewidth(0.8)

    # -- Blended transform: x=data coords, y=axes fraction -------------------
    blend_top = mtransforms.blended_transform_factory(ax.transData, ax.transAxes)

    # -- Condition sub-labels: fixed 5 pt above top spine --------------------
    for j, (cl, key, label) in enumerate(comparisons):
        color = (col_colors or {}).get(key, 'black')
        ax.text(j, 1.0, label,
                transform=blend_top + _pt_up(5),
                ha='center', va='bottom',
                fontsize=font_sz, color=color,
                clip_on=False)

    # -- Cell-line group headers ----------------------------------------------
    if cl_labels:
        cl_runs = [
            (cl, [j for j, _ in group])
            for cl, group in groupby(enumerate(comparisons), key=lambda x: x[1][0])
        ]
        for cl, indices in cl_runs:
            x_mid   = (indices[0] + indices[-1]) / 2
            x_l     = indices[0]  - 0.35
            x_r     = indices[-1] + 0.35
            display = cl_labels.get(cl, cl)

            ax.text(x_mid, 1.0, display,
                    transform=blend_top + _pt_up(cl_header_pts),
                    ha='center', va='bottom',
                    fontsize=font_sz, fontweight='bold',
                    clip_on=False)

            ax.plot([x_l, x_r], [1.0, 1.0],
                    transform=blend_top + _pt_up(cl_header_pts - 5),
                    color='black', lw=0.8,
                    clip_on=False)

    # -- Title ----------------------------------------------------------------
    title_pts = (cl_header_pts + title_gap_pts) if cl_labels else 10
    ax.text(0.5, 1.0, title,
            transform=ax.transAxes + _pt_up(title_pts),
            ha='center', va='bottom',
            fontsize=font_sz, fontweight='bold',
            clip_on=False)

    # -- Colorbar -------------------------------------------------------------
    sm = ScalarMappable(norm=norm, cmap='coolwarm')
    sm.set_array([])
    fig.colorbar(sm, cax=cax)
    cax.set_ylabel('log$_2$ fold change', fontsize=legend_font_sz)
    cax.tick_params(labelsize=legend_font_sz)

    # -- Size legend ----------------------------------------------------------
    p_levels    = sorted(set([0.1, 1e-2, 1e-3, 1e-4, size_cap_p]))
    leg_handles = []
    for p in p_levels:
        lp  = -np.log10(p)
        s   = np.clip((lp / fixed_max_lp) * max_dot, 2, max_dot)
        ms  = 2 * np.sqrt(s / np.pi)
        leg_handles.append(
            Line2D([0], [0], marker='o', color='w',
                   markerfacecolor='grey', markeredgecolor='none',
                   markersize=ms, label=f'p = {p:.0e}')
        )

    # -- Group colour legend --------------------------------------------------
    if use_label_colors:
        shown_groups = {
            gene_to_group[g] for g in gene_order
            if gene_to_group.get(g) in gene_label_colors
        }
        color_handles = [
            Line2D([0], [0], marker='s', color='w',
                   markerfacecolor=gene_label_colors[grp], markeredgecolor='none',
                   markersize=8, label=grp)
            for grp in gene_label_colors
            if grp in shown_groups
        ]
        color_leg = lax.legend(
            handles=color_handles,
            title='Gene group', title_fontsize=font_sz,
            loc='upper center',
            frameon=False, fontsize=font_sz,
            handletextpad=0.5, labelspacing=0.8,
        )
        lax.add_artist(color_leg)
        size_leg_loc = 'lower center'
    else:
        size_leg_loc = 'upper center'

    # bbox_to_anchor lets us slide the legend horizontally within lax:
    # size_legend_x=0.5 is centered (old behavior); push it toward 1.0+ to
    # move it right, e.g. 0.5 -> mostly-right -> fully outside the panel.
    size_leg_anchor_y = 1.0 if size_leg_loc == 'upper center' else 0.0
    lax.legend(handles=leg_handles,
               title='Adjusted\np-value', title_fontsize=legend_font_sz,
               loc=size_leg_loc,
               bbox_to_anchor=(size_legend_x, size_leg_anchor_y),
               bbox_transform=lax.transAxes,
               frameon=False, fontsize=legend_font_sz,
               handletextpad=1.0, labelspacing=1.2)

    # -- Layout ---------------------------------------------------------------
    plt.tight_layout()

    if show_group_labels and group_mids and not use_label_colors:
        fig.subplots_adjust(left=fig.subplotpars.left + 0.12)

    top_margin_in = (title_pts + 14) / 72
    fig.subplots_adjust(
        top=fig.subplotpars.top - top_margin_in / figsize[1]
    )

    pos = cax.get_position()
    cax.set_position([pos.x0, pos.y0, pos.width * 0.25, pos.height])

    if save_path:
        plt.savefig(save_path, bbox_inches='tight')

    plt.show()
    return fig, ax


def gene_level_table(
    results,
    gene_groups,
    comparisons,
    sort_by_significance=False,
    rank_method='min_p',   # NEW: 'min_p' (default, prior behavior) or 'mean_p_penalized'
    top_n=None,
    save_path=None,
):
    """
    Companion to gene_level_dotplot: produces a table of the same genes,
    in the same order, with log2FoldChange and padj for each comparison.

    See gene_level_dotplot / _rank_genes for details on rank_method options.

    Returns a DataFrame with columns:
        gene, <label>_log2FC, <label>_padj, <label>_log2FC, <label>_padj, ...
    one (log2FC, padj) pair per entry in `comparisons`, in the order given.
    """

    # -- Build flat gene list (mirrors gene_level_dotplot) --------------------
    if isinstance(gene_groups, dict):
        gene_order = []
        for genes in gene_groups.values():
            gene_order.extend(genes)
    else:
        gene_order = list(gene_groups)

    n_genes = len(gene_order)
    n_comps = len(comparisons)

    # -- Extract LFC and padj matrices (mirrors gene_level_dotplot) -----------
    lfc_mat  = np.full((n_genes, n_comps), np.nan)
    padj_mat = np.full((n_genes, n_comps), np.nan)

    for j, (cl, key, _) in enumerate(comparisons):
        df = results.get(cl, {}).get(key, pd.DataFrame())
        if df.empty:
            continue
        for i, gene in enumerate(gene_order):
            if gene in df.index:
                lfc_mat[i, j]  = df.loc[gene, 'f_mpv_log2FoldChange']
                padj_mat[i, j] = df.loc[gene, 'f_mpv_padj']

    # -- Sort by significance and apply top_n (mirrors gene_level_dotplot) ----
    if sort_by_significance:
        sort_idx = _rank_genes(padj_mat, rank_method=rank_method)
        if top_n is not None:
            sort_idx = sort_idx[:top_n]
        gene_order = [gene_order[i] for i in sort_idx]
        lfc_mat    = lfc_mat[sort_idx]
        padj_mat   = padj_mat[sort_idx]
        n_genes    = len(gene_order)

    # -- Assemble output dataframe ---------------------------------------------
    out = pd.DataFrame({'gene': gene_order})
    for j, (cl, key, label) in enumerate(comparisons):
        out[f'{label}_log2FC'] = lfc_mat[:, j]
        out[f'{label}_padj']   = padj_mat[:, j]

    if save_path:
        out.to_csv(save_path, sep='\t' if save_path.endswith('.tsv') else ',', index=False)

    return out

In [ ]:
############# exploratory pathway gene group creation ##############################

### first merge IPA results into a dictionary so genes can be found in both easily
results_all_ipa = (
    {f"1806_{k}": v for k, v in results_1806.items()} |
    {f"231_{k}":  v for k, v in results_231.items()}
)

def build_gene_groups(
    ipa_dict,
    results,
    comparisons,
    pathway_groups,          # OrderedDict: {display_name: [pathway, ...], ...}
    min_padj    = 0.1,
    min_lfc     = 0.3,
    remove_overlap = True,   # remove genes already assigned to earlier groups
):
    """
    Build a gene_groups dict ready for gene_level_dotplot.

    Parameters
    ----------
    ipa_dict        : merged IPA results dict (e.g. results_all_ipa)
    results         : DESeq results dict keyed by (cell_line, comparison)
    comparisons     : list of (cell_line, comp, label) tuples
    pathway_groups  : OrderedDict mapping display name -> list of IPA pathway names
    min_padj        : adjusted p-value cutoff for signal filter
    min_lfc         : minimum |log2FC| for signal filter
    remove_overlap  : if True, genes assigned to an earlier group are removed
                      from all later groups (first-group priority)

    Returns
    -------
    gene_groups : dict mapping display name -> filtered gene list
    """

    def _extract(pathway_names):
        genes = set()
        for label, ipa in ipa_dict.items():
            for pathway in pathway_names:
                if pathway in ipa.pathways.index:
                    row = ipa.pathways.loc[pathway]
                    new_genes = [g.strip() for g in row['Molecules'].split(',')]
                    genes.update(new_genes)
        return sorted(genes)

    def _filter(gene_list):
        keep = []
        for gene in gene_list:
            for cell_line, comp, label in comparisons:
                df = results[cell_line][comp]
                if gene in df.index:
                    row = df.loc[gene]
                    if (pd.notna(row['f_mpv_padj'])
                            and row['f_mpv_padj'] < min_padj
                            and abs(row['f_mpv_log2FoldChange']) >= min_lfc):
                        keep.append(gene)
                        break
        return keep

    gene_groups  = {}
    assigned     = set()   # tracks genes already placed in an earlier group

    for display_name, pathway_names in pathway_groups.items():
        # 1. extract
        genes = _extract(pathway_names)

        # 2. remove overlap with previously assigned groups
        if remove_overlap:
            genes = [g for g in genes if g not in assigned]

        # 3. filter by signal
        genes = _filter(genes)

        gene_groups[display_name] = genes
        assigned.update(genes)

        print(f"{display_name}: {len(genes)} genes - {genes}")

    return gene_groups

In [ ]:
# -- 1. Load DESeq2-derived MPV classification results from TSV files --------

de_dir = '/stor/work/Brock/kennedy/SC_repo/data/GeneDosageAnalysis/deseq_data'  # adjust path if needed

results = {}
for fname in os.listdir(de_dir):
    # Only pick up the per-clone MPV classification files, ignore
    # compiled/GSEA/ORA outputs that live in the same folder
    if not fname.endswith('_MPV_classification.tsv'):
        continue

    # e.g. "HCC1806_F_C2C4_MPV_classification.tsv"
    #      -> cl_display = "HCC1806", clone_pair = "C2C4"
    stem = fname.replace('_MPV_classification.tsv', '')  # "HCC1806_F_C2C4"
    parts = stem.split('_')

    cl_display = parts[0]                 # "HCC1806" or "MDAMB231"
    fusion_flag = parts[1]                # "F" (sanity-check below)
    clone_pair = '_'.join(parts[2:])      # "C2C4" (handles any extra underscores)

    if fusion_flag != 'F':
        # unexpected naming, skip rather than silently mis-key
        print(f"Skipping unexpected filename format: {fname}")
        continue

    cl = '1806' if '1806' in cl_display else '231'

    results.setdefault(cl, {})[clone_pair] = pd.read_csv(
        os.path.join(de_dir, fname), sep='\t', index_col=0
    )

In [ ]:
def get_clone_labels(gene_table, comparisons=None):
    """Infer clone labels from a gene_table's columns, or take them from `comparisons`."""
    if comparisons is not None:
        return [label for _, _, label in comparisons]
    return [c[:-len('_log2FC')] for c in gene_table.columns if c.endswith('_log2FC')]


def sig_gene_counts(gene_table, clone_labels=None, padj_thresh=0.05, min_abs_lfc=None):
    """Count significant genes per clone. min_abs_lfc=0.585 matches the official DE-gene cutoff instead of the padj-only significance the dotplots use."""
    if clone_labels is None:
        clone_labels = get_clone_labels(gene_table)
    rows = []
    for label in clone_labels:
        lfc  = gene_table[f'{label}_log2FC']
        padj = gene_table[f'{label}_padj']
        sig = (padj < padj_thresh) & lfc.notna()
        if min_abs_lfc is not None:
            sig &= lfc.abs() >= min_abs_lfc
        rows.append({'clone': label, 'n_sig': int(sig.sum()),
                      'n_up': int((sig & (lfc > 0)).sum()),
                      'n_down': int((sig & (lfc < 0)).sum())})
    return pd.DataFrame(rows).set_index('clone')


def pairwise_concordance(gene_table, clone_a, clone_b, padj_thresh=0.05, min_abs_lfc=None):
    """Genes significant in BOTH clones, and % that move the same direction."""
    lfc_a, padj_a = gene_table[f'{clone_a}_log2FC'], gene_table[f'{clone_a}_padj']
    lfc_b, padj_b = gene_table[f'{clone_b}_log2FC'], gene_table[f'{clone_b}_padj']
    sig_a = (padj_a < padj_thresh) & lfc_a.notna()
    sig_b = (padj_b < padj_thresh) & lfc_b.notna()
    if min_abs_lfc is not None:
        sig_a &= lfc_a.abs() >= min_abs_lfc
        sig_b &= lfc_b.abs() >= min_abs_lfc
    n_a, n_b = int(sig_a.sum()), int(sig_b.sum())
    overlap_mask = sig_a & sig_b
    n_overlap = int(overlap_mask.sum())
    if n_overlap == 0:
        n_concordant, pct_concordant = 0, np.nan
    else:
        same_dir = np.sign(lfc_a[overlap_mask]) == np.sign(lfc_b[overlap_mask])
        n_concordant = int(same_dir.sum())
        pct_concordant = 100 * n_concordant / n_overlap
    return {'clone_a': clone_a, 'clone_b': clone_b, 'n_sig_a': n_a, 'n_sig_b': n_b,
            'n_overlap': n_overlap,
            'pct_overlap_of_smaller': 100 * n_overlap / min(n_a, n_b) if min(n_a, n_b) > 0 else np.nan,
            'n_concordant': n_concordant, 'pct_concordant': pct_concordant}


def all_pairwise_concordance(gene_table, clone_labels=None, padj_thresh=0.05, min_abs_lfc=None):
    """Run pairwise_concordance() over every clone pair; returns a tidy DataFrame."""
    if clone_labels is None:
        clone_labels = get_clone_labels(gene_table)
    rows = [pairwise_concordance(gene_table, a, b, padj_thresh=padj_thresh, min_abs_lfc=min_abs_lfc)
            for a, b in combinations(clone_labels, 2)]
    return pd.DataFrame(rows)


def compare_sig_counts_across_groups(gene_tables_dict, padj_thresh=0.05, min_abs_lfc=None):
    """clone x pathway-group matrix of n_sig - checks 'C6C7/C6C8 biggest across nearly every category'."""
    cols = {name: sig_gene_counts(tbl, padj_thresh=padj_thresh, min_abs_lfc=min_abs_lfc)['n_sig']
            for name, tbl in gene_tables_dict.items()}
    return pd.concat(cols, axis=1)

### Translation group top pathways

In [ ]:
### looking at entire pathway gene sets
comparisons = [
    ("1806", "C2C4", "C2C4"),
    ("1806", "C5C2", "C5C2"),
    ("1806", "C6C7", "C6C7"),
    ("231", "C1C4", "C1C4"),
    ("231", "C6C8", "C6C8"),
]

pathway_groups = OrderedDict({
    'Translation / RiBi': [
        'Major pathway of rRNA processing in the nucleolus and cytosol',
        'Eukaryotic Translation Initiation',
        # 'Eukaryotic Translation Elongation',
        # 'Eukaryotic Translation Termination',
        'SRP-dependent cotranslational protein targeting to membrane',
        # 'Nonsense-Mediated Decay (NMD)',
        # 'Ribosomal Quality Control Signaling Pathway',
        # 'Response of EIF2AK4 (GCN2) to amino acid deficiency',
        # 'rRNA modification in the nucleus and cytosol',
        # 'Regulation of eIF4 and p70S6K Signaling',
        # 'Selenoamino acid metabolism',
        # 'Exosome Signaling Pathway',
        # 'Regulation of mRNA stability by proteins that bind AU-rich elements',
    ],
})

story_6_genes = build_gene_groups(
    ipa_dict       = results_all_ipa,
    results        = results,
    comparisons    = comparisons,
    pathway_groups = pathway_groups,
    min_padj       = 0.05,
    min_lfc        = 0.3,
    remove_overlap = True,
)


fig, ax = gene_level_dotplot(
    results     = results,
    gene_groups = story_6_genes,
    comparisons = comparisons,
    # col_colors  = {
    #     "Fusion_vs_Control": "#FF8C00",
    #     # "MS_vs_Control":     "#5E80E5",
    #     # "CF_vs_Control":     "#1A8D8D"
    # },
    cl_labels   = {"1806": "HCC1806", "231": "MDA-MB-231"},
    title       = '  ',
    lfc_range   = 0.65,
    size_cap_p  = 1e-5,
    save_path   = 'plot_transl.svg',
    show_group_labels=True,
    sort_by_significance = True,
    rank_method='mean_p_penalized',
    top_n                = 20,
    ylabel      = "Translation / RiBi (Top Genes from Top Pathways)",
    col_spacing=0.35,
    row_height=0.32,
    size_legend_x=0.8,
    legend_font_sz=14,
)

transl_table = gene_level_table(
    results     = results,
    gene_groups = story_6_genes,
    comparisons = comparisons,
    sort_by_significance = True,
    top_n                = None,
    save_path   = 'genes_transl.tsv',
)
print(transl_table)

### Cell Cycle group top pathways

In [ ]:
### looking at entire pathway gene sets
comparisons = [
    ("1806", "C2C4", "C2C4"),
    ("1806", "C5C2", "C5C2"),
    ("1806", "C6C7", "C6C7"),
    ("231", "C1C4", "C1C4"),
    ("231", "C6C8", "C6C8"),
]

pathway_groups = OrderedDict({
    # 'Translation / RiBi': [
    #     'Major pathway of rRNA processing in the nucleolus and cytosol',
    #     'Eukaryotic Translation Initiation',
    #     'Eukaryotic Translation Elongation',
    #     'Eukaryotic Translation Termination',
    #     'SRP-dependent cotranslational protein targeting to membrane',
    #     'Nonsense-Mediated Decay (NMD)',
    #     'Ribosomal Quality Control Signaling Pathway',
    #     'Response of EIF2AK4 (GCN2) to amino acid deficiency',
    #     'rRNA modification in the nucleus and cytosol',
    #     'Regulation of eIF4 and p70S6K Signaling',
    #     'Selenoamino acid metabolism',
    #     'Exosome Signaling Pathway',
    #     'Regulation of mRNA stability by proteins that bind AU-rich elements',
    # ],
    'Cell Cycle': [
        'Mitotic G2-G2/M phases',
        # 'Mitotic G1 phase and G1/S transition',
        # 'S Phase',
        'Mitotic Metaphase and Anaphase',
        'Cell Cycle Checkpoints',
        # 'Regulation of mitotic cell cycle',
        # 'DNA Replication Pre-Initiation',
        # 'Synthesis of DNA',
        # 'Cell Cycle: G2/M DNA Damage Checkpoint Regulation',
    ],
    # 'Proteostasis / Protein Turnover': [
    #     'Proteasome assembly',
    #     'Protein Ubiquitination Pathway',
    #     'Neddylation',
    #     'NIK-->noncanonical NF-kB signaling',
    #     'Deubiquitination',
    #     'Degradation of beta-catenin by the destruction complex',
    #     'Degradation of CRY and PER proteins',
    #     'TNFR2 non-canonical NF-kB pathway',
    #     'HSP90 chaperone cycle for steroid hormone receptor',
    #     'Protein folding',
    #     'Post-translational protein phosphorylation',
    # ],
    # 'Mitochondria / OXPHOS': [
    #     'Oxidative Phosphorylation',
    #     'Respiratory electron transport',
    #     'Cristae formation',
    #     'Mitochondrial translation',
    #     'Mitochondrial protein degradation',
    #     'tRNA processing in the mitochondrion',
    #     'Complex IV assembly',
    #     'Mitochondrial protein import',
    # ],
    # 'DNA Damage Response & Repair': [
    #     'Nucleotide Excision Repair',
    #     'NER (Nucleotide Excision Repair, Enhanced Pathway)',
    #     'Resolution of Abasic Sites (AP sites)',
    #     'BER (Base Excision Repair) Pathway',
    #     'HDR through MMEJ (alt-NHEJ)',
    #     'Role of CHK Proteins in Cell Cycle Checkpoint Control',
    #     'DNA damage-induced 14-3-3σ Signaling',
    #     'DNA Damage/Telomere Stress Induced Senescence',
    # ],
    # 'RNA Processing / Modification': [
    #     'RNA m6A Methylation Signaling Pathway',
    #     'Metabolism of non-coding RNA',
    #     'Processing of Capped Intron-Containing Pre-mRNA',
    #     'RNA Polymerase II Transcription',
    #     'mRNA Capping',
    #     'RNA polymerase II transcribes snRNA genes',
    #     'Maternal to Zygotic Transition Signaling Pathway',
    #     'XIST Epigenetic Regulation Signaling Pathway',
    # ],
    # 'Hedgehog / Developmental Signaling': [
    #     'Hedgehog ligand biogenesis',
    #     'Hedgehog \'off\' state',
    #     'Hedgehog \'on\' state',
    #     'Signaling by NOTCH4',
    #     'TCF dependent signaling in response to WNT',
    #     'Transcriptional activity of SMAD2/SMAD3:SMAD4 heterotrimer',
    #     'Signaling by TGF-beta Receptor Complex',
    #     'Cardiac Valve Formation Signaling Pathway',
    #     'Cerebral Malformation Signaling Pathway',
    #     'Somitogenesis',
    #     'Mouse Embryonic Stem Cell Pluripotency',
    # ],
    # 'Cytoskeleton / Rho-GTPase / MAPK Signaling': [
    #     'RHO GTPase cycle',
    #     'RHOGDI Signaling',
    #     'Signaling by Rho Family GTPases',
    #     'RHO GTPases Activate Formins',
    #     'RHO GTPases activate IQGAPs',
    #     'RAC Signaling',
    #     'Actin Cytoskeleton Signaling',
    #     'Actin Nucleation by ARP-WASP Complex',
    #     'Paxillin Signaling',
    #     'ILK Signaling',
    #     'ERK/MAPK Signaling',
    #     'RAF/MAP kinase cascade',
    #     'RAF-independent MAPK1/3 activation',
    #     'MAPK6/MAPK4 signaling',
    #     'Ephrin Receptor Signaling',
    #     'Integrin to Cytoskeleton Signaling Pathway',
    #     'Cell junction organization',
    #     'Clathrin-mediated endocytosis',
    #     'Regulation of Cellular Mechanics by Calpain Protease',
    #     'Neuron Navigator Signaling Pathway',
    # ],
    # 'Interferon / Antiviral Immune': [
    #     'Interferon alpha/beta signaling',
    #     'Interferon gamma signaling',
    #     'Antigen Presentation Pathway',
    #     'Modulation of host responses by IFN-stimulated genes',
    #     'Role of PKR in Interferon Induction and Antiviral Response',
    #     'Role of RIG1-like Receptors in Antiviral Innate Immunity',
    #     'Cytosolic sensors of pathogen-associated DNA',
    #     'GAIT Translation Signaling Pathway',
    # ],
    # 'Cytokine / Inflammatory Signaling': [
    #     'IL-8 Signaling',
    #     'Chemokine Signaling',
    #     'Other interleukin signaling',
    #     'Interleukin-1 family signaling',
    #     'Role of IL-17F in Allergic Inflammatory Airway Diseases',
    #     'TNFR1 Signaling',
    #     'TNFR2 Signaling',
    #     'TCR signaling',
    #     'Signaling by the B Cell Receptor (BCR)',
    #     'Fc epsilon receptor (FCERI) signaling',
    #     'B Cell Activating Factor Signaling',
    #     'C-type lectin receptors (CLRs)',
    #     'Neutrophil degranulation',
    #     'Oncostatin M Signaling',
    #     'fMLP Signaling in Neutrophils',
    #     'Interleukin-10 signaling',
    #     'Role of Macrophages, Fibroblasts and Endothelial Cells in Rheumatoid Arthritis',
    #     'Acute Phase Response Signaling',
    #     'Tumor Microenvironment Pathway',
    #     'Hematoma Resolution Signaling Pathway',
    #     'Gene and protein expression by JAK-STAT signaling',
    # ],
    # 'ECM / Fibrosis / Collagen': [
    #     'Assembly of collagen fibrils and other multimeric structures',
    #     'Collagen biosynthesis and modifying enzymes',
    #     'Extracellular matrix organization',
    #     'Hepatic Fibrosis Signaling Pathway',
    #     'Pulmonary Fibrosis Idiopathic Signaling Pathway',
    #     'Formation of the dystrophin-glycoprotein complex',
    #     'Keratinization',
    # ],
    # 'Cellular Stress Response': [
    #     'Cellular response to heat stress',
    #     'Cellular response to hypoxia',
    #     'XBP1(S) activates chaperone genes',
    #     'NFE2L2 regulating anti-oxidant/detoxification enzymes',
    #     'KEAP1-NFE2L2 pathway',
    #     'Cachexia Signaling Pathway',
    #     'Response of EIF2AK1 (HRI) to heme deficiency',
    # ],
    # 'Hormone / Nuclear Receptor Signaling': [
    #     'ESR-mediated signaling',
    #     'Extra-nuclear estrogen signaling',
    #     'TR/RXR Activation',
    #     'VDR/RXR Activation',
    #     'Oxytocin Signaling Pathway',
    #     'Oxytocin in Brain Signaling Pathway',
    #     'α-Adrenergic Signaling',
    # ],
    # 'Growth Factor / RTK Signaling': [
    #     'IGF-1 Signaling',
    #     'Regulation of Insulin-like Growth Factor (IGF) transport and uptake by IGFBPs',
    #     'NGF Signaling',
    #     'MSP-RON Signaling in Cancer Cells Pathway',
    #     'Signaling by MET',
    #     'GP6 Signaling Pathway',
    #     'Thrombin Signaling',
    #     'Signaling by VEGF',
    # ],
    # 'Other': [
    #     'ABC-family proteins mediated transport',
    #     'ABRA Signaling Pathway',
    #     'Apoptotic execution phase',
    #     'Centrosomal KIAA0586 Signaling Pathway',
    #     'Cilia Biogenesis Signaling Pathway',
    #     'DYRK1A Signaling Pathway',
    #     'Epithelial Membrane Protein Signaling Pathway',
    #     'Folate Signaling Pathway',
    #     'G-Protein Coupled Receptor Signaling',
    #     'Glycation Signaling Pathway',
    #     'Hereditary Breast Cancer Signaling',
    #     'Interconversion of nucleotide di- and triphosphates',
    #     'L1CAM interactions',
    #     'Metabolism of polyamines',
    #     'Microautophagy Signaling Pathway',
    #     'Myelination Signaling Pathway',
    #     'Parkinson\'s Signaling Pathway',
    #     'Plasma lipoprotein assembly, remodeling, and clearance',
    #     'Regulation of Apoptosis',
    #     'Regulation of RUNX2 expression and activity',
    #     'Role of Tissue Factor in Cancer',
    #     'Sertoli Cell-Germ Cell Junction Signaling Pathway',
    #     'Sertoli Cell-Sertoli Cell Junction Signaling',
    #     'Signaling by Hippo',
    #     'Signaling by ROBO receptors',
    #     'Transcriptional regulation by RUNX1',
    #     'Transcriptional regulation by RUNX3',
    #     'Type II Diabetes Mellitus Signaling',
    # ],
})

story_6_genes = build_gene_groups(
    ipa_dict       = results_all_ipa,
    results        = results,
    comparisons    = comparisons,
    pathway_groups = pathway_groups,
    min_padj       = 0.05,
    min_lfc        = 0.3,
    remove_overlap = True,
)


fig, ax = gene_level_dotplot(
    results     = results,
    gene_groups = story_6_genes,
    comparisons = comparisons,
    # col_colors  = {
    #     "Fusion_vs_Control": "#FF8C00",
    #     # "MS_vs_Control":     "#5E80E5",
    #     # "CF_vs_Control":     "#1A8D8D"
    # },
    cl_labels   = {"1806": "HCC1806", "231": "MDA-MB-231"},
    title       = '  ',
    lfc_range   = 0.65,
    size_cap_p  = 1e-5,
    save_path   = 'plot_cellcycle.svg',
    show_group_labels=True,
    sort_by_significance = True,
    rank_method='mean_p_penalized',
    top_n                = 20,
    ylabel      = "Cell Cycle (Top Genes from Top Pathways)",
    col_spacing=0.35,
    row_height=0.32,
    size_legend_x=0.8,
    legend_font_sz=14,
    )

cellcycle_table = gene_level_table(
    results     = results,
    gene_groups = story_6_genes,
    comparisons = comparisons,
    sort_by_significance = True,
    top_n                = None,
    save_path   = 'genes_cellcycle.tsv',
)
print(cellcycle_table)

### Mitochondria group top pathways

In [ ]:
### looking at entire pathway gene sets
comparisons = [
    ("1806", "C2C4", "C2C4"),
    ("1806", "C5C2", "C5C2"),
    ("1806", "C6C7", "C6C7"),
    ("231", "C1C4", "C1C4"),
    ("231", "C6C8", "C6C8"),
]

pathway_groups = OrderedDict({
    # 'Translation / RiBi': [
    #     'Major pathway of rRNA processing in the nucleolus and cytosol',
    #     'Eukaryotic Translation Initiation',
    #     'Eukaryotic Translation Elongation',
    #     'Eukaryotic Translation Termination',
    #     'SRP-dependent cotranslational protein targeting to membrane',
    #     'Nonsense-Mediated Decay (NMD)',
    #     'Ribosomal Quality Control Signaling Pathway',
    #     'Response of EIF2AK4 (GCN2) to amino acid deficiency',
    #     'rRNA modification in the nucleus and cytosol',
    #     'Regulation of eIF4 and p70S6K Signaling',
    #     'Selenoamino acid metabolism',
    #     'Exosome Signaling Pathway',
    #     'Regulation of mRNA stability by proteins that bind AU-rich elements',
    # ],
    # 'Cell Cycle': [
    #     'Mitotic G2-G2/M phases',
    #     'Mitotic G1 phase and G1/S transition',
    #     'S Phase',
    #     'Mitotic Metaphase and Anaphase',
    #     'Cell Cycle Checkpoints',
    #     'Regulation of mitotic cell cycle',
    #     'DNA Replication Pre-Initiation',
    #     'Synthesis of DNA',
    #     'Cell Cycle: G2/M DNA Damage Checkpoint Regulation',
    # ],
    # 'Proteostasis / Protein Turnover': [
    #     'Proteasome assembly',
    #     'Protein Ubiquitination Pathway',
    #     'Neddylation',
    #     'NIK-->noncanonical NF-kB signaling',
    #     'Deubiquitination',
    #     'Degradation of beta-catenin by the destruction complex',
    #     'Degradation of CRY and PER proteins',
    #     'TNFR2 non-canonical NF-kB pathway',
    #     'HSP90 chaperone cycle for steroid hormone receptor',
    #     'Protein folding',
    #     'Post-translational protein phosphorylation',
    # ],
    'Mitochondria / OXPHOS': [
        'Oxidative Phosphorylation',
        'Respiratory electron transport',
        # 'Cristae formation',
        # 'Mitochondrial translation',
        # 'Mitochondrial protein degradation',
        # 'tRNA processing in the mitochondrion',
        # 'Complex IV assembly',
        'Mitochondrial protein import',
    ],
    # 'DNA Damage Response & Repair': [
    #     'Nucleotide Excision Repair',
    #     'NER (Nucleotide Excision Repair, Enhanced Pathway)',
    #     'Resolution of Abasic Sites (AP sites)',
    #     'BER (Base Excision Repair) Pathway',
    #     'HDR through MMEJ (alt-NHEJ)',
    #     'Role of CHK Proteins in Cell Cycle Checkpoint Control',
    #     'DNA damage-induced 14-3-3σ Signaling',
    #     'DNA Damage/Telomere Stress Induced Senescence',
    # ],
    # 'RNA Processing / Modification': [
    #     'RNA m6A Methylation Signaling Pathway',
    #     'Metabolism of non-coding RNA',
    #     'Processing of Capped Intron-Containing Pre-mRNA',
    #     'RNA Polymerase II Transcription',
    #     'mRNA Capping',
    #     'RNA polymerase II transcribes snRNA genes',
    #     'Maternal to Zygotic Transition Signaling Pathway',
    #     'XIST Epigenetic Regulation Signaling Pathway',
    # ],
    # 'Hedgehog / Developmental Signaling': [
    #     'Hedgehog ligand biogenesis',
    #     'Hedgehog \'off\' state',
    #     'Hedgehog \'on\' state',
    #     'Signaling by NOTCH4',
    #     'TCF dependent signaling in response to WNT',
    #     'Transcriptional activity of SMAD2/SMAD3:SMAD4 heterotrimer',
    #     'Signaling by TGF-beta Receptor Complex',
    #     'Cardiac Valve Formation Signaling Pathway',
    #     'Cerebral Malformation Signaling Pathway',
    #     'Somitogenesis',
    #     'Mouse Embryonic Stem Cell Pluripotency',
    # ],
    # 'Cytoskeleton / Rho-GTPase / MAPK Signaling': [
    #     'RHO GTPase cycle',
    #     'RHOGDI Signaling',
    #     'Signaling by Rho Family GTPases',
    #     'RHO GTPases Activate Formins',
    #     'RHO GTPases activate IQGAPs',
    #     'RAC Signaling',
    #     'Actin Cytoskeleton Signaling',
    #     'Actin Nucleation by ARP-WASP Complex',
    #     'Paxillin Signaling',
    #     'ILK Signaling',
    #     'ERK/MAPK Signaling',
    #     'RAF/MAP kinase cascade',
    #     'RAF-independent MAPK1/3 activation',
    #     'MAPK6/MAPK4 signaling',
    #     'Ephrin Receptor Signaling',
    #     'Integrin to Cytoskeleton Signaling Pathway',
    #     'Cell junction organization',
    #     'Clathrin-mediated endocytosis',
    #     'Regulation of Cellular Mechanics by Calpain Protease',
    #     'Neuron Navigator Signaling Pathway',
    # ],
    # 'Interferon / Antiviral Immune': [
    #     'Interferon alpha/beta signaling',
    #     'Interferon gamma signaling',
    #     'Antigen Presentation Pathway',
    #     'Modulation of host responses by IFN-stimulated genes',
    #     'Role of PKR in Interferon Induction and Antiviral Response',
    #     'Role of RIG1-like Receptors in Antiviral Innate Immunity',
    #     'Cytosolic sensors of pathogen-associated DNA',
    #     'GAIT Translation Signaling Pathway',
    # ],
    # 'Cytokine / Inflammatory Signaling': [
    #     'IL-8 Signaling',
    #     'Chemokine Signaling',
    #     'Other interleukin signaling',
    #     'Interleukin-1 family signaling',
    #     'Role of IL-17F in Allergic Inflammatory Airway Diseases',
    #     'TNFR1 Signaling',
    #     'TNFR2 Signaling',
    #     'TCR signaling',
    #     'Signaling by the B Cell Receptor (BCR)',
    #     'Fc epsilon receptor (FCERI) signaling',
    #     'B Cell Activating Factor Signaling',
    #     'C-type lectin receptors (CLRs)',
    #     'Neutrophil degranulation',
    #     'Oncostatin M Signaling',
    #     'fMLP Signaling in Neutrophils',
    #     'Interleukin-10 signaling',
    #     'Role of Macrophages, Fibroblasts and Endothelial Cells in Rheumatoid Arthritis',
    #     'Acute Phase Response Signaling',
    #     'Tumor Microenvironment Pathway',
    #     'Hematoma Resolution Signaling Pathway',
    #     'Gene and protein expression by JAK-STAT signaling',
    # ],
    # 'ECM / Fibrosis / Collagen': [
    #     'Assembly of collagen fibrils and other multimeric structures',
    #     'Collagen biosynthesis and modifying enzymes',
    #     'Extracellular matrix organization',
    #     'Hepatic Fibrosis Signaling Pathway',
    #     'Pulmonary Fibrosis Idiopathic Signaling Pathway',
    #     'Formation of the dystrophin-glycoprotein complex',
    #     'Keratinization',
    # ],
    # 'Cellular Stress Response': [
    #     'Cellular response to heat stress',
    #     'Cellular response to hypoxia',
    #     'XBP1(S) activates chaperone genes',
    #     'NFE2L2 regulating anti-oxidant/detoxification enzymes',
    #     'KEAP1-NFE2L2 pathway',
    #     'Cachexia Signaling Pathway',
    #     'Response of EIF2AK1 (HRI) to heme deficiency',
    # ],
    # 'Hormone / Nuclear Receptor Signaling': [
    #     'ESR-mediated signaling',
    #     'Extra-nuclear estrogen signaling',
    #     'TR/RXR Activation',
    #     'VDR/RXR Activation',
    #     'Oxytocin Signaling Pathway',
    #     'Oxytocin in Brain Signaling Pathway',
    #     'α-Adrenergic Signaling',
    # ],
    # 'Growth Factor / RTK Signaling': [
    #     'IGF-1 Signaling',
    #     'Regulation of Insulin-like Growth Factor (IGF) transport and uptake by IGFBPs',
    #     'NGF Signaling',
    #     'MSP-RON Signaling in Cancer Cells Pathway',
    #     'Signaling by MET',
    #     'GP6 Signaling Pathway',
    #     'Thrombin Signaling',
    #     'Signaling by VEGF',
    # ],
    # 'Other': [
    #     'ABC-family proteins mediated transport',
    #     'ABRA Signaling Pathway',
    #     'Apoptotic execution phase',
    #     'Centrosomal KIAA0586 Signaling Pathway',
    #     'Cilia Biogenesis Signaling Pathway',
    #     'DYRK1A Signaling Pathway',
    #     'Epithelial Membrane Protein Signaling Pathway',
    #     'Folate Signaling Pathway',
    #     'G-Protein Coupled Receptor Signaling',
    #     'Glycation Signaling Pathway',
    #     'Hereditary Breast Cancer Signaling',
    #     'Interconversion of nucleotide di- and triphosphates',
    #     'L1CAM interactions',
    #     'Metabolism of polyamines',
    #     'Microautophagy Signaling Pathway',
    #     'Myelination Signaling Pathway',
    #     'Parkinson\'s Signaling Pathway',
    #     'Plasma lipoprotein assembly, remodeling, and clearance',
    #     'Regulation of Apoptosis',
    #     'Regulation of RUNX2 expression and activity',
    #     'Role of Tissue Factor in Cancer',
    #     'Sertoli Cell-Germ Cell Junction Signaling Pathway',
    #     'Sertoli Cell-Sertoli Cell Junction Signaling',
    #     'Signaling by Hippo',
    #     'Signaling by ROBO receptors',
    #     'Transcriptional regulation by RUNX1',
    #     'Transcriptional regulation by RUNX3',
    #     'Type II Diabetes Mellitus Signaling',
    # ],
})

story_6_genes = build_gene_groups(
    ipa_dict       = results_all_ipa,
    results        = results,
    comparisons    = comparisons,
    pathway_groups = pathway_groups,
    min_padj       = 0.05,
    min_lfc        = 0.3,
    remove_overlap = True,
)


fig, ax = gene_level_dotplot(
    results     = results,
    gene_groups = story_6_genes,
    comparisons = comparisons,
    # col_colors  = {
    #     "Fusion_vs_Control": "#FF8C00",
    #     # "MS_vs_Control":     "#5E80E5",
    #     # "CF_vs_Control":     "#1A8D8D"
    # },
    cl_labels   = {"1806": "HCC1806", "231": "MDA-MB-231"},
    title       = '  ',
    lfc_range   = 0.65,
    size_cap_p  = 1e-5,
    save_path   = 'plot_mito.svg',
    show_group_labels=True,
    sort_by_significance = True,
    rank_method='mean_p_penalized',
    top_n                = 20,
    ylabel      = "Mitochondria / OXPHOS (Top Genes from Top Pathways)",
    col_spacing=0.35,
    row_height=0.32,
    size_legend_x=0.8,
    legend_font_sz=14,
)

mito_table = gene_level_table(
    results     = results,
    gene_groups = story_6_genes,
    comparisons = comparisons,
    sort_by_significance = True,
    top_n                = None,
    save_path   = 'genes_mito.tsv',
)
print(mito_table)

### Proteostasis group top pathways

In [ ]:
### looking at entire pathway gene sets
comparisons = [
    ("1806", "C2C4", "C2C4"),
    ("1806", "C5C2", "C5C2"),
    ("1806", "C6C7", "C6C7"),
    ("231", "C1C4", "C1C4"),
    ("231", "C6C8", "C6C8"),
]

pathway_groups = OrderedDict({
    # 'Translation / RiBi': [
    #     'Major pathway of rRNA processing in the nucleolus and cytosol',
    #     'Eukaryotic Translation Initiation',
    #     'Eukaryotic Translation Elongation',
    #     'Eukaryotic Translation Termination',
    #     'SRP-dependent cotranslational protein targeting to membrane',
    #     'Nonsense-Mediated Decay (NMD)',
    #     'Ribosomal Quality Control Signaling Pathway',
    #     'Response of EIF2AK4 (GCN2) to amino acid deficiency',
    #     'rRNA modification in the nucleus and cytosol',
    #     'Regulation of eIF4 and p70S6K Signaling',
    #     'Selenoamino acid metabolism',
    #     'Exosome Signaling Pathway',
    #     'Regulation of mRNA stability by proteins that bind AU-rich elements',
    # ],
    # 'Cell Cycle': [
    #     'Mitotic G2-G2/M phases',
    #     'Mitotic G1 phase and G1/S transition',
    #     'S Phase',
    #     'Mitotic Metaphase and Anaphase',
    #     'Cell Cycle Checkpoints',
    #     'Regulation of mitotic cell cycle',
    #     'DNA Replication Pre-Initiation',
    #     'Synthesis of DNA',
    #     'Cell Cycle: G2/M DNA Damage Checkpoint Regulation',
    # ],
    'Proteostasis / Protein Turnover': [
        # 'Proteasome assembly',
        'Protein Ubiquitination Pathway',
        'Neddylation',
        # 'NIK-->noncanonical NF-kB signaling',
        # 'Deubiquitination',
        'Degradation of beta-catenin by the destruction complex',
        # 'Degradation of CRY and PER proteins',
        # 'TNFR2 non-canonical NF-kB pathway',
        # 'HSP90 chaperone cycle for steroid hormone receptor',
        # 'Protein folding',
        # 'Post-translational protein phosphorylation',
    ],
    # 'Mitochondria / OXPHOS': [
    #     'Oxidative Phosphorylation',
    #     'Respiratory electron transport',
    #     'Cristae formation',
    #     'Mitochondrial translation',
    #     'Mitochondrial protein degradation',
    #     'tRNA processing in the mitochondrion',
    #     'Complex IV assembly',
    #     'Mitochondrial protein import',
    # ],
    # 'DNA Damage Response & Repair': [
    #     'Nucleotide Excision Repair',
    #     'NER (Nucleotide Excision Repair, Enhanced Pathway)',
    #     'Resolution of Abasic Sites (AP sites)',
    #     'BER (Base Excision Repair) Pathway',
    #     'HDR through MMEJ (alt-NHEJ)',
    #     'Role of CHK Proteins in Cell Cycle Checkpoint Control',
    #     'DNA damage-induced 14-3-3σ Signaling',
    #     'DNA Damage/Telomere Stress Induced Senescence',
    # ],
    # 'RNA Processing / Modification': [
    #     'RNA m6A Methylation Signaling Pathway',
    #     'Metabolism of non-coding RNA',
    #     'Processing of Capped Intron-Containing Pre-mRNA',
    #     'RNA Polymerase II Transcription',
    #     'mRNA Capping',
    #     'RNA polymerase II transcribes snRNA genes',
    #     'Maternal to Zygotic Transition Signaling Pathway',
    #     'XIST Epigenetic Regulation Signaling Pathway',
    # ],
    # 'Hedgehog / Developmental Signaling': [
    #     'Hedgehog ligand biogenesis',
    #     'Hedgehog \'off\' state',
    #     'Hedgehog \'on\' state',
    #     'Signaling by NOTCH4',
    #     'TCF dependent signaling in response to WNT',
    #     'Transcriptional activity of SMAD2/SMAD3:SMAD4 heterotrimer',
    #     'Signaling by TGF-beta Receptor Complex',
    #     'Cardiac Valve Formation Signaling Pathway',
    #     'Cerebral Malformation Signaling Pathway',
    #     'Somitogenesis',
    #     'Mouse Embryonic Stem Cell Pluripotency',
    # ],
    # 'Cytoskeleton / Rho-GTPase / MAPK Signaling': [
    #     'RHO GTPase cycle',
    #     'RHOGDI Signaling',
    #     'Signaling by Rho Family GTPases',
    #     'RHO GTPases Activate Formins',
    #     'RHO GTPases activate IQGAPs',
    #     'RAC Signaling',
    #     'Actin Cytoskeleton Signaling',
    #     'Actin Nucleation by ARP-WASP Complex',
    #     'Paxillin Signaling',
    #     'ILK Signaling',
    #     'ERK/MAPK Signaling',
    #     'RAF/MAP kinase cascade',
    #     'RAF-independent MAPK1/3 activation',
    #     'MAPK6/MAPK4 signaling',
    #     'Ephrin Receptor Signaling',
    #     'Integrin to Cytoskeleton Signaling Pathway',
    #     'Cell junction organization',
    #     'Clathrin-mediated endocytosis',
    #     'Regulation of Cellular Mechanics by Calpain Protease',
    #     'Neuron Navigator Signaling Pathway',
    # ],
    # 'Interferon / Antiviral Immune': [
    #     'Interferon alpha/beta signaling',
    #     'Interferon gamma signaling',
    #     'Antigen Presentation Pathway',
    #     'Modulation of host responses by IFN-stimulated genes',
    #     'Role of PKR in Interferon Induction and Antiviral Response',
    #     'Role of RIG1-like Receptors in Antiviral Innate Immunity',
    #     'Cytosolic sensors of pathogen-associated DNA',
    #     'GAIT Translation Signaling Pathway',
    # ],
    # 'Cytokine / Inflammatory Signaling': [
    #     'IL-8 Signaling',
    #     'Chemokine Signaling',
    #     'Other interleukin signaling',
    #     'Interleukin-1 family signaling',
    #     'Role of IL-17F in Allergic Inflammatory Airway Diseases',
    #     'TNFR1 Signaling',
    #     'TNFR2 Signaling',
    #     'TCR signaling',
    #     'Signaling by the B Cell Receptor (BCR)',
    #     'Fc epsilon receptor (FCERI) signaling',
    #     'B Cell Activating Factor Signaling',
    #     'C-type lectin receptors (CLRs)',
    #     'Neutrophil degranulation',
    #     'Oncostatin M Signaling',
    #     'fMLP Signaling in Neutrophils',
    #     'Interleukin-10 signaling',
    #     'Role of Macrophages, Fibroblasts and Endothelial Cells in Rheumatoid Arthritis',
    #     'Acute Phase Response Signaling',
    #     'Tumor Microenvironment Pathway',
    #     'Hematoma Resolution Signaling Pathway',
    #     'Gene and protein expression by JAK-STAT signaling',
    # ],
    # 'ECM / Fibrosis / Collagen': [
    #     'Assembly of collagen fibrils and other multimeric structures',
    #     'Collagen biosynthesis and modifying enzymes',
    #     'Extracellular matrix organization',
    #     'Hepatic Fibrosis Signaling Pathway',
    #     'Pulmonary Fibrosis Idiopathic Signaling Pathway',
    #     'Formation of the dystrophin-glycoprotein complex',
    #     'Keratinization',
    # ],
    # 'Cellular Stress Response': [
    #     'Cellular response to heat stress',
    #     'Cellular response to hypoxia',
    #     'XBP1(S) activates chaperone genes',
    #     'NFE2L2 regulating anti-oxidant/detoxification enzymes',
    #     'KEAP1-NFE2L2 pathway',
    #     'Cachexia Signaling Pathway',
    #     'Response of EIF2AK1 (HRI) to heme deficiency',
    # ],
    # 'Hormone / Nuclear Receptor Signaling': [
    #     'ESR-mediated signaling',
    #     'Extra-nuclear estrogen signaling',
    #     'TR/RXR Activation',
    #     'VDR/RXR Activation',
    #     'Oxytocin Signaling Pathway',
    #     'Oxytocin in Brain Signaling Pathway',
    #     'α-Adrenergic Signaling',
    # ],
    # 'Growth Factor / RTK Signaling': [
    #     'IGF-1 Signaling',
    #     'Regulation of Insulin-like Growth Factor (IGF) transport and uptake by IGFBPs',
    #     'NGF Signaling',
    #     'MSP-RON Signaling in Cancer Cells Pathway',
    #     'Signaling by MET',
    #     'GP6 Signaling Pathway',
    #     'Thrombin Signaling',
    #     'Signaling by VEGF',
    # ],
    # 'Other': [
    #     'ABC-family proteins mediated transport',
    #     'ABRA Signaling Pathway',
    #     'Apoptotic execution phase',
    #     'Centrosomal KIAA0586 Signaling Pathway',
    #     'Cilia Biogenesis Signaling Pathway',
    #     'DYRK1A Signaling Pathway',
    #     'Epithelial Membrane Protein Signaling Pathway',
    #     'Folate Signaling Pathway',
    #     'G-Protein Coupled Receptor Signaling',
    #     'Glycation Signaling Pathway',
    #     'Hereditary Breast Cancer Signaling',
    #     'Interconversion of nucleotide di- and triphosphates',
    #     'L1CAM interactions',
    #     'Metabolism of polyamines',
    #     'Microautophagy Signaling Pathway',
    #     'Myelination Signaling Pathway',
    #     'Parkinson\'s Signaling Pathway',
    #     'Plasma lipoprotein assembly, remodeling, and clearance',
    #     'Regulation of Apoptosis',
    #     'Regulation of RUNX2 expression and activity',
    #     'Role of Tissue Factor in Cancer',
    #     'Sertoli Cell-Germ Cell Junction Signaling Pathway',
    #     'Sertoli Cell-Sertoli Cell Junction Signaling',
    #     'Signaling by Hippo',
    #     'Signaling by ROBO receptors',
    #     'Transcriptional regulation by RUNX1',
    #     'Transcriptional regulation by RUNX3',
    #     'Type II Diabetes Mellitus Signaling',
    # ],
})

story_6_genes = build_gene_groups(
    ipa_dict       = results_all_ipa,
    results        = results,
    comparisons    = comparisons,
    pathway_groups = pathway_groups,
    min_padj       = 0.05,
    min_lfc        = 0.3,
    remove_overlap = True,
)


fig, ax = gene_level_dotplot(
    results     = results,
    gene_groups = story_6_genes,
    comparisons = comparisons,
    # col_colors  = {
    #     "Fusion_vs_Control": "#FF8C00",
    #     # "MS_vs_Control":     "#5E80E5",
    #     # "CF_vs_Control":     "#1A8D8D"
    # },
    cl_labels   = {"1806": "HCC1806", "231": "MDA-MB-231"},
    title       = '  ',
    lfc_range   = 0.65,
    size_cap_p  = 1e-5,
    save_path   = 'plot_proteo.svg',
    show_group_labels=True,
    sort_by_significance = True,
    rank_method='mean_p_penalized',
    top_n                = 20,
    ylabel      = "Proteostasis / Protein Turnover (Top Genes from Top Pathways)",
    col_spacing=0.35,
    row_height=0.32,
    size_legend_x=0.8,
    legend_font_sz=14,
)

proteo_table = gene_level_table(
    results     = results,
    gene_groups = story_6_genes,
    comparisons = comparisons,
    sort_by_significance = True,
    top_n                = None,
    save_path   = 'genes_proteo.tsv',
)
print(proteo_table)

### Interferon/antiviral top pathways

In [ ]:
### looking at entire pathway gene sets
comparisons = [
    ("1806", "C2C4", "C2C4"),
    ("1806", "C5C2", "C5C2"),
    ("1806", "C6C7", "C6C7"),
    ("231", "C1C4", "C1C4"),
    ("231", "C6C8", "C6C8"),
]

pathway_groups = OrderedDict({
    # 'Translation / RiBi': [
    #     "Major pathway of rRNA processing in the nucleolus and cytosol",
    #     "Eukaryotic Translation Initiation",
    #     "Eukaryotic Translation Elongation",
    #     "Eukaryotic Translation Termination",
    #     "SRP-dependent cotranslational protein targeting to membrane",
    #     "Nonsense-Mediated Decay (NMD)",
    #     "Ribosomal Quality Control Signaling Pathway",
    #     "Response of EIF2AK4 (GCN2) to amino acid deficiency",
    #     "rRNA modification in the nucleus and cytosol",
    #     "Regulation of eIF4 and p70S6K Signaling",
    #     "Selenoamino acid metabolism",
    #     "Exosome Signaling Pathway",
    #     "Regulation of mRNA stability by proteins that bind AU-rich elements",
    #     # "Signaling by ROBO receptors",
    # ],
    # 'Cell Cycle': [
    #     "Mitotic G2-G2/M phases",
    #     "Mitotic G1 phase and G1/S transition",
    #     "S Phase",
    #     "Mitotic Metaphase and Anaphase",
    #     "Cell Cycle Checkpoints",
    #     "Regulation of mitotic cell cycle",
    #     "DNA Replication Pre-Initiation",
    #     "Synthesis of DNA",
    # ],
    # 'Proteostasis / Protein Turnover': [
    #     "Proteasome assembly",
    #     "Protein Ubiquitination Pathway",
    #     "Neddylation",
    #     "NIK-->noncanonical NF-kB signaling",
    #     "Deubiquitination",
    #     "Degradation of beta-catenin by the destruction complex",
    #     "Degradation of CRY and PER proteins",
    #     "TNFR2 non-canonical NF-kB pathway",
    #     "HSP90 chaperone cycle for steroid hormone receptor",
    #     "Protein folding",
    # ],
    # 'Mitochondria / OXPHOS': [
    #     "Oxidative Phosphorylation",
    #     "Respiratory electron transport",
    #     "Cristae formation",
    #     "Mitochondrial translation",
    #     "Mitochondrial protein degradation",
    #     "tRNA processing in the mitochondrion",
    #     "Complex IV assembly",
    # ],
    # 'DNA Damage Response & Repair': [
    #     "Nucleotide Excision Repair",
    #     "NER (Nucleotide Excision Repair, Enhanced Pathway)",
    #     "Resolution of Abasic Sites (AP sites)",
    #     "BER (Base Excision Repair) Pathway",
    #     "HDR through MMEJ (alt-NHEJ)",
    #     "Role of CHK Proteins in Cell Cycle Checkpoint Control",
    #     "DNA damage-induced 14-3-3σ Signaling",
    #     "DNA Damage/Telomere Stress Induced Senescence",
    # ],
    # 'RNA Modification': [
    #     "RNA m6A Methylation Signaling Pathway",
    #     "Metabolism of non-coding RNA",
    # ],
    # 'Hedgehog / Developmental Signaling': [
    #     "Hedgehog ligand biogenesis",
    #     "Hedgehog 'off' state",
    #     "Hedgehog 'on' state",
    #     "Signaling by NOTCH4",
    #     "TCF dependent signaling in response to WNT",
    #     "Transcriptional activity of SMAD2/SMAD3:SMAD4 heterotrimer",
    #     "Signaling by TGF-beta Receptor Complex",
    # ],
    # 'Cytoskeleton / Rho-GTPase / MAPK Signaling': [
    #     "RHO GTPase cycle",
    #     "RHOGDI Signaling",
    #     "Signaling by Rho Family GTPases",
    #     "RHO GTPases Activate Formins",
    #     "RHO GTPases activate IQGAPs",
    #     "RAC Signaling",
    #     "Actin Cytoskeleton Signaling",
    #     "Actin Nucleation by ARP-WASP Complex",
    #     "Paxillin Signaling",
    #     "ILK Signaling",
    #     "ERK/MAPK Signaling",
    #     "RAF/MAP kinase cascade",
    #     "RAF-independent MAPK1/3 activation",
    #     "MAPK6/MAPK4 signaling",
    #     "Ephrin Receptor Signaling",
    #     "Integrin to Cytoskeleton Signaling Pathway",
    #     "Cell junction organization",
    # ],
    'Interferon / Antiviral Immune': [
        "Interferon alpha/beta signaling",
        # "Interferon gamma signaling",
        # "Antigen Presentation Pathway",
        # "Modulation of host responses by IFN-stimulated genes",
        "Role of PKR in Interferon Induction and Antiviral Response",
        # "Role of RIG1-like Receptors in Antiviral Innate Immunity",
        # "Cytosolic sensors of pathogen-associated DNA",
        "GAIT Translation Signaling Pathway",
    ],
    # 'Cytokine / Inflammatory Signaling': [
    #     "IL-8 Signaling",
    #     "Chemokine Signaling",
    #     "Other interleukin signaling",
    #     "Interleukin-1 family signaling",
    #     "Role of IL-17F in Allergic Inflammatory Airway Diseases",
    #     "TNFR1 Signaling",
    #     "TNFR2 Signaling",
    #     "TCR signaling",
    #     "Signaling by the B Cell Receptor (BCR)",
    #     "Fc epsilon receptor (FCERI) signaling",
    #     "B Cell Activating Factor Signaling",
    #     "C-type lectin receptors (CLRs)",
    #     "Neutrophil degranulation",
    #     "Oncostatin M Signaling",
    #     "fMLP Signaling in Neutrophils",
    # ],
})

story_6_genes = build_gene_groups(
    ipa_dict       = results_all_ipa,
    results        = results,
    comparisons    = comparisons,
    pathway_groups = pathway_groups,
    min_padj       = 0.05,
    min_lfc        = 0.3,
    remove_overlap = True,
)


fig, ax = gene_level_dotplot(
    results     = results,
    gene_groups = story_6_genes,
    comparisons = comparisons,
    # col_colors  = {
    #     "Fusion_vs_Control": "#FF8C00",
    #     # "MS_vs_Control":     "#5E80E5",
    #     # "CF_vs_Control":     "#1A8D8D"
    # },
    cl_labels   = {"1806": "HCC1806", "231": "MDA-MB-231"},
    title       = '  ',
    lfc_range   = 0.65,
    size_cap_p  = 1e-5,
    save_path   = 'plot_interferon.svg',
    show_group_labels=True,
    sort_by_significance = True,
    rank_method='mean_p_penalized',
    top_n                = 20,
    ylabel      = "Interferon / Antiviral Immune (Top Genes from Top Pathways)",
    col_spacing=0.35,
    row_height=0.32,
    size_legend_x=0.8,
    legend_font_sz=14,
)

interferon_table = gene_level_table(
    results     = results,
    gene_groups = story_6_genes,
    comparisons = comparisons,
    sort_by_significance = True,
    top_n                = None,
    save_path   = 'genes_interferon.tsv',
)
print(interferon_table)

### RNA processing/modification

In [ ]:
### looking at entire pathway gene sets
comparisons = [
    ("1806", "C2C4", "C2C4"),
    ("1806", "C5C2", "C5C2"),
    ("1806", "C6C7", "C6C7"),
    ("231", "C1C4", "C1C4"),
    ("231", "C6C8", "C6C8"),
]

pathway_groups = OrderedDict({
    # 'Translation / RiBi': [
    #     'Major pathway of rRNA processing in the nucleolus and cytosol',
    #     'Eukaryotic Translation Initiation',
    #     'Eukaryotic Translation Elongation',
    #     'Eukaryotic Translation Termination',
    #     'SRP-dependent cotranslational protein targeting to membrane',
    #     'Nonsense-Mediated Decay (NMD)',
    #     'Ribosomal Quality Control Signaling Pathway',
    #     'Response of EIF2AK4 (GCN2) to amino acid deficiency',
    #     'rRNA modification in the nucleus and cytosol',
    #     'Regulation of eIF4 and p70S6K Signaling',
    #     'Selenoamino acid metabolism',
    #     'Exosome Signaling Pathway',
    #     'Regulation of mRNA stability by proteins that bind AU-rich elements',
    # ],
    # 'Cell Cycle': [
    #     'Mitotic G2-G2/M phases',
    #     'Mitotic G1 phase and G1/S transition',
    #     'S Phase',
    #     'Mitotic Metaphase and Anaphase',
    #     'Cell Cycle Checkpoints',
    #     'Regulation of mitotic cell cycle',
    #     'DNA Replication Pre-Initiation',
    #     'Synthesis of DNA',
    #     'Cell Cycle: G2/M DNA Damage Checkpoint Regulation',
    # ],
    # 'Proteostasis / Protein Turnover': [
    #     'Proteasome assembly',
    #     'Protein Ubiquitination Pathway',
    #     'Neddylation',
    #     'NIK-->noncanonical NF-kB signaling',
    #     'Deubiquitination',
    #     'Degradation of beta-catenin by the destruction complex',
    #     'Degradation of CRY and PER proteins',
    #     'TNFR2 non-canonical NF-kB pathway',
    #     'HSP90 chaperone cycle for steroid hormone receptor',
    #     'Protein folding',
    #     'Post-translational protein phosphorylation',
    # ],
    # 'Mitochondria / OXPHOS': [
    #     'Oxidative Phosphorylation',
    #     'Respiratory electron transport',
    #     'Cristae formation',
    #     'Mitochondrial translation',
    #     'Mitochondrial protein degradation',
    #     'tRNA processing in the mitochondrion',
    #     'Complex IV assembly',
    #     'Mitochondrial protein import',
    # ],
    # 'DNA Damage Response & Repair': [
    #     'Nucleotide Excision Repair',
    #     'NER (Nucleotide Excision Repair, Enhanced Pathway)',
    #     'Resolution of Abasic Sites (AP sites)',
    #     'BER (Base Excision Repair) Pathway',
    #     'HDR through MMEJ (alt-NHEJ)',
    #     'Role of CHK Proteins in Cell Cycle Checkpoint Control',
    #     'DNA damage-induced 14-3-3σ Signaling',
    #     'DNA Damage/Telomere Stress Induced Senescence',
    # ],
    'RNA Processing / Modification': [
        'RNA m6A Methylation Signaling Pathway',
        # 'Metabolism of non-coding RNA',
        'Processing of Capped Intron-Containing Pre-mRNA',
        'RNA Polymerase II Transcription',
        # 'mRNA Capping',
        # 'RNA polymerase II transcribes snRNA genes',
        # 'Maternal to Zygotic Transition Signaling Pathway',
        # 'XIST Epigenetic Regulation Signaling Pathway',
    ],
    # 'Hedgehog / Developmental Signaling': [
    #     'Hedgehog ligand biogenesis',
    #     'Hedgehog \'off\' state',
    #     'Hedgehog \'on\' state',
    #     'Signaling by NOTCH4',
    #     'TCF dependent signaling in response to WNT',
    #     'Transcriptional activity of SMAD2/SMAD3:SMAD4 heterotrimer',
    #     'Signaling by TGF-beta Receptor Complex',
    #     'Cardiac Valve Formation Signaling Pathway',
    #     'Cerebral Malformation Signaling Pathway',
    #     'Somitogenesis',
    #     'Mouse Embryonic Stem Cell Pluripotency',
    # ],
    # 'Cytoskeleton / Rho-GTPase / MAPK Signaling': [
    #     'RHO GTPase cycle',
    #     'RHOGDI Signaling',
    #     'Signaling by Rho Family GTPases',
    #     'RHO GTPases Activate Formins',
    #     'RHO GTPases activate IQGAPs',
    #     'RAC Signaling',
    #     'Actin Cytoskeleton Signaling',
    #     'Actin Nucleation by ARP-WASP Complex',
    #     'Paxillin Signaling',
    #     'ILK Signaling',
    #     'ERK/MAPK Signaling',
    #     'RAF/MAP kinase cascade',
    #     'RAF-independent MAPK1/3 activation',
    #     'MAPK6/MAPK4 signaling',
    #     'Ephrin Receptor Signaling',
    #     'Integrin to Cytoskeleton Signaling Pathway',
    #     'Cell junction organization',
    #     'Clathrin-mediated endocytosis',
    #     'Regulation of Cellular Mechanics by Calpain Protease',
    #     'Neuron Navigator Signaling Pathway',
    # ],
    # 'Interferon / Antiviral Immune': [
    #     'Interferon alpha/beta signaling',
    #     'Interferon gamma signaling',
    #     'Antigen Presentation Pathway',
    #     'Modulation of host responses by IFN-stimulated genes',
    #     'Role of PKR in Interferon Induction and Antiviral Response',
    #     'Role of RIG1-like Receptors in Antiviral Innate Immunity',
    #     'Cytosolic sensors of pathogen-associated DNA',
    #     'GAIT Translation Signaling Pathway',
    # ],
    # 'Cytokine / Inflammatory Signaling': [
    #     'IL-8 Signaling',
    #     'Chemokine Signaling',
    #     'Other interleukin signaling',
    #     'Interleukin-1 family signaling',
    #     'Role of IL-17F in Allergic Inflammatory Airway Diseases',
    #     'TNFR1 Signaling',
    #     'TNFR2 Signaling',
    #     'TCR signaling',
    #     'Signaling by the B Cell Receptor (BCR)',
    #     'Fc epsilon receptor (FCERI) signaling',
    #     'B Cell Activating Factor Signaling',
    #     'C-type lectin receptors (CLRs)',
    #     'Neutrophil degranulation',
    #     'Oncostatin M Signaling',
    #     'fMLP Signaling in Neutrophils',
    #     'Interleukin-10 signaling',
    #     'Role of Macrophages, Fibroblasts and Endothelial Cells in Rheumatoid Arthritis',
    #     'Acute Phase Response Signaling',
    #     'Tumor Microenvironment Pathway',
    #     'Hematoma Resolution Signaling Pathway',
    #     'Gene and protein expression by JAK-STAT signaling',
    # ],
    # 'ECM / Fibrosis / Collagen': [
    #     'Assembly of collagen fibrils and other multimeric structures',
    #     'Collagen biosynthesis and modifying enzymes',
    #     'Extracellular matrix organization',
    #     'Hepatic Fibrosis Signaling Pathway',
    #     'Pulmonary Fibrosis Idiopathic Signaling Pathway',
    #     'Formation of the dystrophin-glycoprotein complex',
    #     'Keratinization',
    # ],
    # 'Cellular Stress Response': [
    #     'Cellular response to heat stress',
    #     'Cellular response to hypoxia',
    #     'XBP1(S) activates chaperone genes',
    #     'NFE2L2 regulating anti-oxidant/detoxification enzymes',
    #     'KEAP1-NFE2L2 pathway',
    #     'Cachexia Signaling Pathway',
    #     'Response of EIF2AK1 (HRI) to heme deficiency',
    # ],
    # 'Hormone / Nuclear Receptor Signaling': [
    #     'ESR-mediated signaling',
    #     'Extra-nuclear estrogen signaling',
    #     'TR/RXR Activation',
    #     'VDR/RXR Activation',
    #     'Oxytocin Signaling Pathway',
    #     'Oxytocin in Brain Signaling Pathway',
    #     'α-Adrenergic Signaling',
    # ],
    # 'Growth Factor / RTK Signaling': [
    #     'IGF-1 Signaling',
    #     'Regulation of Insulin-like Growth Factor (IGF) transport and uptake by IGFBPs',
    #     'NGF Signaling',
    #     'MSP-RON Signaling in Cancer Cells Pathway',
    #     'Signaling by MET',
    #     'GP6 Signaling Pathway',
    #     'Thrombin Signaling',
    #     'Signaling by VEGF',
    # ],
    # 'Other': [
    #     'ABC-family proteins mediated transport',
    #     'ABRA Signaling Pathway',
    #     'Apoptotic execution phase',
    #     'Centrosomal KIAA0586 Signaling Pathway',
    #     'Cilia Biogenesis Signaling Pathway',
    #     'DYRK1A Signaling Pathway',
    #     'Epithelial Membrane Protein Signaling Pathway',
    #     'Folate Signaling Pathway',
    #     'G-Protein Coupled Receptor Signaling',
    #     'Glycation Signaling Pathway',
    #     'Hereditary Breast Cancer Signaling',
    #     'Interconversion of nucleotide di- and triphosphates',
    #     'L1CAM interactions',
    #     'Metabolism of polyamines',
    #     'Microautophagy Signaling Pathway',
    #     'Myelination Signaling Pathway',
    #     'Parkinson\'s Signaling Pathway',
    #     'Plasma lipoprotein assembly, remodeling, and clearance',
    #     'Regulation of Apoptosis',
    #     'Regulation of RUNX2 expression and activity',
    #     'Role of Tissue Factor in Cancer',
    #     'Sertoli Cell-Germ Cell Junction Signaling Pathway',
    #     'Sertoli Cell-Sertoli Cell Junction Signaling',
    #     'Signaling by Hippo',
    #     'Signaling by ROBO receptors',
    #     'Transcriptional regulation by RUNX1',
    #     'Transcriptional regulation by RUNX3',
    #     'Type II Diabetes Mellitus Signaling',
    # ],
})

story_6_genes = build_gene_groups(
    ipa_dict       = results_all_ipa,
    results        = results,
    comparisons    = comparisons,
    pathway_groups = pathway_groups,
    min_padj       = 0.05,
    min_lfc        = 0.3,
    remove_overlap = True,
)


fig, ax = gene_level_dotplot(
    results     = results,
    gene_groups = story_6_genes,
    comparisons = comparisons,
    # col_colors  = {
    #     "Fusion_vs_Control": "#FF8C00",
    #     # "MS_vs_Control":     "#5E80E5",
    #     # "CF_vs_Control":     "#1A8D8D"
    # },
    cl_labels   = {"1806": "HCC1806", "231": "MDA-MB-231"},
    title       = '  ',
    lfc_range   = 0.65,
    size_cap_p  = 1e-5,
    save_path   = 'plot_test.svg',
    col_spacing = 0.25,
    show_group_labels=True,
    sort_by_significance = True,
    rank_method='mean_p_penalized',
    top_n                = 20,
    ylabel      = "RNA Processing / Modification (Top Genes from Top Pathways)",
)

rnaproc_table = gene_level_table(
    results     = results,
    gene_groups = story_6_genes,
    comparisons = comparisons,
    sort_by_significance = True,
    top_n                = None,
    save_path   = 'genes_rnaproc.tsv',
)
print(rnaproc_table)

### DNA damage response/repair

In [ ]:
### looking at entire pathway gene sets
comparisons = [
    ("1806", "C2C4", "C2C4"),
    ("1806", "C5C2", "C5C2"),
    ("1806", "C6C7", "C6C7"),
    ("231", "C1C4", "C1C4"),
    ("231", "C6C8", "C6C8"),
]

pathway_groups = OrderedDict({
    # 'Translation / RiBi': [
    #     'Major pathway of rRNA processing in the nucleolus and cytosol',
    #     'Eukaryotic Translation Initiation',
    #     'Eukaryotic Translation Elongation',
    #     'Eukaryotic Translation Termination',
    #     'SRP-dependent cotranslational protein targeting to membrane',
    #     'Nonsense-Mediated Decay (NMD)',
    #     'Ribosomal Quality Control Signaling Pathway',
    #     'Response of EIF2AK4 (GCN2) to amino acid deficiency',
    #     'rRNA modification in the nucleus and cytosol',
    #     'Regulation of eIF4 and p70S6K Signaling',
    #     'Selenoamino acid metabolism',
    #     'Exosome Signaling Pathway',
    #     'Regulation of mRNA stability by proteins that bind AU-rich elements',
    # ],
    # 'Cell Cycle': [
    #     'Mitotic G2-G2/M phases',
    #     'Mitotic G1 phase and G1/S transition',
    #     'S Phase',
    #     'Mitotic Metaphase and Anaphase',
    #     'Cell Cycle Checkpoints',
    #     'Regulation of mitotic cell cycle',
    #     'DNA Replication Pre-Initiation',
    #     'Synthesis of DNA',
    #     'Cell Cycle: G2/M DNA Damage Checkpoint Regulation',
    # ],
    # 'Proteostasis / Protein Turnover': [
    #     'Proteasome assembly',
    #     'Protein Ubiquitination Pathway',
    #     'Neddylation',
    #     'NIK-->noncanonical NF-kB signaling',
    #     'Deubiquitination',
    #     'Degradation of beta-catenin by the destruction complex',
    #     'Degradation of CRY and PER proteins',
    #     'TNFR2 non-canonical NF-kB pathway',
    #     'HSP90 chaperone cycle for steroid hormone receptor',
    #     'Protein folding',
    #     'Post-translational protein phosphorylation',
    # ],
    # 'Mitochondria / OXPHOS': [
    #     'Oxidative Phosphorylation',
    #     'Respiratory electron transport',
    #     'Cristae formation',
    #     'Mitochondrial translation',
    #     'Mitochondrial protein degradation',
    #     'tRNA processing in the mitochondrion',
    #     'Complex IV assembly',
    #     'Mitochondrial protein import',
    # ],
    'DNA Damage Response & Repair': [
        'Nucleotide Excision Repair',
        'NER (Nucleotide Excision Repair, Enhanced Pathway)',
        # 'Resolution of Abasic Sites (AP sites)',
        # 'BER (Base Excision Repair) Pathway',
        # 'HDR through MMEJ (alt-NHEJ)',
        'Role of CHK Proteins in Cell Cycle Checkpoint Control',
        # 'DNA damage-induced 14-3-3σ Signaling',
        # 'DNA Damage/Telomere Stress Induced Senescence',
    ],
    # 'RNA Processing / Modification': [
        # 'RNA m6A Methylation Signaling Pathway',
        # 'Metabolism of non-coding RNA',
        # 'Processing of Capped Intron-Containing Pre-mRNA',
        # 'RNA Polymerase II Transcription',
        # 'mRNA Capping',
        # 'RNA polymerase II transcribes snRNA genes',
        # 'Maternal to Zygotic Transition Signaling Pathway',
        # 'XIST Epigenetic Regulation Signaling Pathway',
    # ],
    # 'Hedgehog / Developmental Signaling': [
    #     'Hedgehog ligand biogenesis',
    #     'Hedgehog \'off\' state',
    #     'Hedgehog \'on\' state',
    #     'Signaling by NOTCH4',
    #     'TCF dependent signaling in response to WNT',
    #     'Transcriptional activity of SMAD2/SMAD3:SMAD4 heterotrimer',
    #     'Signaling by TGF-beta Receptor Complex',
    #     'Cardiac Valve Formation Signaling Pathway',
    #     'Cerebral Malformation Signaling Pathway',
    #     'Somitogenesis',
    #     'Mouse Embryonic Stem Cell Pluripotency',
    # ],
    # 'Cytoskeleton / Rho-GTPase / MAPK Signaling': [
    #     'RHO GTPase cycle',
    #     'RHOGDI Signaling',
    #     'Signaling by Rho Family GTPases',
    #     'RHO GTPases Activate Formins',
    #     'RHO GTPases activate IQGAPs',
    #     'RAC Signaling',
    #     'Actin Cytoskeleton Signaling',
    #     'Actin Nucleation by ARP-WASP Complex',
    #     'Paxillin Signaling',
    #     'ILK Signaling',
    #     'ERK/MAPK Signaling',
    #     'RAF/MAP kinase cascade',
    #     'RAF-independent MAPK1/3 activation',
    #     'MAPK6/MAPK4 signaling',
    #     'Ephrin Receptor Signaling',
    #     'Integrin to Cytoskeleton Signaling Pathway',
    #     'Cell junction organization',
    #     'Clathrin-mediated endocytosis',
    #     'Regulation of Cellular Mechanics by Calpain Protease',
    #     'Neuron Navigator Signaling Pathway',
    # ],
    # 'Interferon / Antiviral Immune': [
    #     'Interferon alpha/beta signaling',
    #     'Interferon gamma signaling',
    #     'Antigen Presentation Pathway',
    #     'Modulation of host responses by IFN-stimulated genes',
    #     'Role of PKR in Interferon Induction and Antiviral Response',
    #     'Role of RIG1-like Receptors in Antiviral Innate Immunity',
    #     'Cytosolic sensors of pathogen-associated DNA',
    #     'GAIT Translation Signaling Pathway',
    # ],
    # 'Cytokine / Inflammatory Signaling': [
    #     'IL-8 Signaling',
    #     'Chemokine Signaling',
    #     'Other interleukin signaling',
    #     'Interleukin-1 family signaling',
    #     'Role of IL-17F in Allergic Inflammatory Airway Diseases',
    #     'TNFR1 Signaling',
    #     'TNFR2 Signaling',
    #     'TCR signaling',
    #     'Signaling by the B Cell Receptor (BCR)',
    #     'Fc epsilon receptor (FCERI) signaling',
    #     'B Cell Activating Factor Signaling',
    #     'C-type lectin receptors (CLRs)',
    #     'Neutrophil degranulation',
    #     'Oncostatin M Signaling',
    #     'fMLP Signaling in Neutrophils',
    #     'Interleukin-10 signaling',
    #     'Role of Macrophages, Fibroblasts and Endothelial Cells in Rheumatoid Arthritis',
    #     'Acute Phase Response Signaling',
    #     'Tumor Microenvironment Pathway',
    #     'Hematoma Resolution Signaling Pathway',
    #     'Gene and protein expression by JAK-STAT signaling',
    # ],
    # 'ECM / Fibrosis / Collagen': [
    #     'Assembly of collagen fibrils and other multimeric structures',
    #     'Collagen biosynthesis and modifying enzymes',
    #     'Extracellular matrix organization',
    #     'Hepatic Fibrosis Signaling Pathway',
    #     'Pulmonary Fibrosis Idiopathic Signaling Pathway',
    #     'Formation of the dystrophin-glycoprotein complex',
    #     'Keratinization',
    # ],
    # 'Cellular Stress Response': [
    #     'Cellular response to heat stress',
    #     'Cellular response to hypoxia',
    #     'XBP1(S) activates chaperone genes',
    #     'NFE2L2 regulating anti-oxidant/detoxification enzymes',
    #     'KEAP1-NFE2L2 pathway',
    #     'Cachexia Signaling Pathway',
    #     'Response of EIF2AK1 (HRI) to heme deficiency',
    # ],
    # 'Hormone / Nuclear Receptor Signaling': [
    #     'ESR-mediated signaling',
    #     'Extra-nuclear estrogen signaling',
    #     'TR/RXR Activation',
    #     'VDR/RXR Activation',
    #     'Oxytocin Signaling Pathway',
    #     'Oxytocin in Brain Signaling Pathway',
    #     'α-Adrenergic Signaling',
    # ],
    # 'Growth Factor / RTK Signaling': [
    #     'IGF-1 Signaling',
    #     'Regulation of Insulin-like Growth Factor (IGF) transport and uptake by IGFBPs',
    #     'NGF Signaling',
    #     'MSP-RON Signaling in Cancer Cells Pathway',
    #     'Signaling by MET',
    #     'GP6 Signaling Pathway',
    #     'Thrombin Signaling',
    #     'Signaling by VEGF',
    # ],
    # 'Other': [
    #     'ABC-family proteins mediated transport',
    #     'ABRA Signaling Pathway',
    #     'Apoptotic execution phase',
    #     'Centrosomal KIAA0586 Signaling Pathway',
    #     'Cilia Biogenesis Signaling Pathway',
    #     'DYRK1A Signaling Pathway',
    #     'Epithelial Membrane Protein Signaling Pathway',
    #     'Folate Signaling Pathway',
    #     'G-Protein Coupled Receptor Signaling',
    #     'Glycation Signaling Pathway',
    #     'Hereditary Breast Cancer Signaling',
    #     'Interconversion of nucleotide di- and triphosphates',
    #     'L1CAM interactions',
    #     'Metabolism of polyamines',
    #     'Microautophagy Signaling Pathway',
    #     'Myelination Signaling Pathway',
    #     'Parkinson\'s Signaling Pathway',
    #     'Plasma lipoprotein assembly, remodeling, and clearance',
    #     'Regulation of Apoptosis',
    #     'Regulation of RUNX2 expression and activity',
    #     'Role of Tissue Factor in Cancer',
    #     'Sertoli Cell-Germ Cell Junction Signaling Pathway',
    #     'Sertoli Cell-Sertoli Cell Junction Signaling',
    #     'Signaling by Hippo',
    #     'Signaling by ROBO receptors',
    #     'Transcriptional regulation by RUNX1',
    #     'Transcriptional regulation by RUNX3',
    #     'Type II Diabetes Mellitus Signaling',
    # ],
})

story_6_genes = build_gene_groups(
    ipa_dict       = results_all_ipa,
    results        = results,
    comparisons    = comparisons,
    pathway_groups = pathway_groups,
    min_padj       = 0.05,
    min_lfc        = 0.3,
    remove_overlap = True,
)


fig, ax = gene_level_dotplot(
    results     = results,
    gene_groups = story_6_genes,
    comparisons = comparisons,
    # col_colors  = {
    #     "Fusion_vs_Control": "#FF8C00",
    #     # "MS_vs_Control":     "#5E80E5",
    #     # "CF_vs_Control":     "#1A8D8D"
    # },
    cl_labels   = {"1806": "HCC1806", "231": "MDA-MB-231"},
    title       = '  ',
    lfc_range   = 0.65,
    size_cap_p  = 1e-5,
    save_path   = 'plot_test.svg',
    col_spacing = 0.25,
    show_group_labels=True,
    sort_by_significance = True,
    rank_method='mean_p_penalized',
    top_n                = 20,
    ylabel      = "dnarepair",
)

gene_table = gene_level_table(
    results     = results,
    gene_groups = story_6_genes,
    comparisons = comparisons,
    sort_by_significance = True,
    top_n                = None,
    save_path   = 'test_dnarepair_genes.tsv',
)
print(gene_table)

### ECM/fibrosis/collagen

In [ ]:
### looking at entire pathway gene sets
comparisons = [
    ("1806", "C2C4", "C2C4"),
    ("1806", "C5C2", "C5C2"),
    ("1806", "C6C7", "C6C7"),
    ("231", "C1C4", "C1C4"),
    ("231", "C6C8", "C6C8"),
]

pathway_groups = OrderedDict({
    # 'Translation / RiBi': [
    #     'Major pathway of rRNA processing in the nucleolus and cytosol',
    #     'Eukaryotic Translation Initiation',
    #     'Eukaryotic Translation Elongation',
    #     'Eukaryotic Translation Termination',
    #     'SRP-dependent cotranslational protein targeting to membrane',
    #     'Nonsense-Mediated Decay (NMD)',
    #     'Ribosomal Quality Control Signaling Pathway',
    #     'Response of EIF2AK4 (GCN2) to amino acid deficiency',
    #     'rRNA modification in the nucleus and cytosol',
    #     'Regulation of eIF4 and p70S6K Signaling',
    #     'Selenoamino acid metabolism',
    #     'Exosome Signaling Pathway',
    #     'Regulation of mRNA stability by proteins that bind AU-rich elements',
    # ],
    # 'Cell Cycle': [
    #     'Mitotic G2-G2/M phases',
    #     'Mitotic G1 phase and G1/S transition',
    #     'S Phase',
    #     'Mitotic Metaphase and Anaphase',
    #     'Cell Cycle Checkpoints',
    #     'Regulation of mitotic cell cycle',
    #     'DNA Replication Pre-Initiation',
    #     'Synthesis of DNA',
    #     'Cell Cycle: G2/M DNA Damage Checkpoint Regulation',
    # ],
    # 'Proteostasis / Protein Turnover': [
    #     'Proteasome assembly',
    #     'Protein Ubiquitination Pathway',
    #     'Neddylation',
    #     'NIK-->noncanonical NF-kB signaling',
    #     'Deubiquitination',
    #     'Degradation of beta-catenin by the destruction complex',
    #     'Degradation of CRY and PER proteins',
    #     'TNFR2 non-canonical NF-kB pathway',
    #     'HSP90 chaperone cycle for steroid hormone receptor',
    #     'Protein folding',
    #     'Post-translational protein phosphorylation',
    # ],
    # 'Mitochondria / OXPHOS': [
    #     'Oxidative Phosphorylation',
    #     'Respiratory electron transport',
    #     'Cristae formation',
    #     'Mitochondrial translation',
    #     'Mitochondrial protein degradation',
    #     'tRNA processing in the mitochondrion',
    #     'Complex IV assembly',
    #     'Mitochondrial protein import',
    # ],
    # 'DNA Damage Response & Repair': [
    #     'Nucleotide Excision Repair',
    #     'NER (Nucleotide Excision Repair, Enhanced Pathway)',
    #     'Resolution of Abasic Sites (AP sites)',
    #     'BER (Base Excision Repair) Pathway',
    #     'HDR through MMEJ (alt-NHEJ)',
    #     'Role of CHK Proteins in Cell Cycle Checkpoint Control',
    #     'DNA damage-induced 14-3-3σ Signaling',
    #     'DNA Damage/Telomere Stress Induced Senescence',
    # ],
    # 'RNA Processing / Modification': [
        # 'RNA m6A Methylation Signaling Pathway',
        # 'Metabolism of non-coding RNA',
        # 'Processing of Capped Intron-Containing Pre-mRNA',
        # 'RNA Polymerase II Transcription',
        # 'mRNA Capping',
        # 'RNA polymerase II transcribes snRNA genes',
        # 'Maternal to Zygotic Transition Signaling Pathway',
        # 'XIST Epigenetic Regulation Signaling Pathway',
    # ],
    # 'Hedgehog / Developmental Signaling': [
    #     'Hedgehog ligand biogenesis',
    #     'Hedgehog \'off\' state',
    #     'Hedgehog \'on\' state',
    #     'Signaling by NOTCH4',
    #     'TCF dependent signaling in response to WNT',
    #     'Transcriptional activity of SMAD2/SMAD3:SMAD4 heterotrimer',
    #     'Signaling by TGF-beta Receptor Complex',
    #     'Cardiac Valve Formation Signaling Pathway',
    #     'Cerebral Malformation Signaling Pathway',
    #     'Somitogenesis',
    #     'Mouse Embryonic Stem Cell Pluripotency',
    # ],
    # 'Cytoskeleton / Rho-GTPase / MAPK Signaling': [
    #     'RHO GTPase cycle',
    #     'RHOGDI Signaling',
    #     'Signaling by Rho Family GTPases',
    #     'RHO GTPases Activate Formins',
    #     'RHO GTPases activate IQGAPs',
    #     'RAC Signaling',
    #     'Actin Cytoskeleton Signaling',
    #     'Actin Nucleation by ARP-WASP Complex',
    #     'Paxillin Signaling',
    #     'ILK Signaling',
    #     'ERK/MAPK Signaling',
    #     'RAF/MAP kinase cascade',
    #     'RAF-independent MAPK1/3 activation',
    #     'MAPK6/MAPK4 signaling',
    #     'Ephrin Receptor Signaling',
    #     'Integrin to Cytoskeleton Signaling Pathway',
    #     'Cell junction organization',
    #     'Clathrin-mediated endocytosis',
    #     'Regulation of Cellular Mechanics by Calpain Protease',
    #     'Neuron Navigator Signaling Pathway',
    # ],
    # 'Interferon / Antiviral Immune': [
    #     'Interferon alpha/beta signaling',
    #     'Interferon gamma signaling',
    #     'Antigen Presentation Pathway',
    #     'Modulation of host responses by IFN-stimulated genes',
    #     'Role of PKR in Interferon Induction and Antiviral Response',
    #     'Role of RIG1-like Receptors in Antiviral Innate Immunity',
    #     'Cytosolic sensors of pathogen-associated DNA',
    #     'GAIT Translation Signaling Pathway',
    # ],
    # 'Cytokine / Inflammatory Signaling': [
    #     'IL-8 Signaling',
    #     'Chemokine Signaling',
    #     'Other interleukin signaling',
    #     'Interleukin-1 family signaling',
    #     'Role of IL-17F in Allergic Inflammatory Airway Diseases',
    #     'TNFR1 Signaling',
    #     'TNFR2 Signaling',
    #     'TCR signaling',
    #     'Signaling by the B Cell Receptor (BCR)',
    #     'Fc epsilon receptor (FCERI) signaling',
    #     'B Cell Activating Factor Signaling',
    #     'C-type lectin receptors (CLRs)',
    #     'Neutrophil degranulation',
    #     'Oncostatin M Signaling',
    #     'fMLP Signaling in Neutrophils',
    #     'Interleukin-10 signaling',
    #     'Role of Macrophages, Fibroblasts and Endothelial Cells in Rheumatoid Arthritis',
    #     'Acute Phase Response Signaling',
    #     'Tumor Microenvironment Pathway',
    #     'Hematoma Resolution Signaling Pathway',
    #     'Gene and protein expression by JAK-STAT signaling',
    # ],
    'ECM / Fibrosis / Collagen': [
        'Assembly of collagen fibrils and other multimeric structures',
        # 'Collagen biosynthesis and modifying enzymes',
        # 'Extracellular matrix organization',
        'Hepatic Fibrosis Signaling Pathway',
        'Pulmonary Fibrosis Idiopathic Signaling Pathway',
        # 'Formation of the dystrophin-glycoprotein complex',
        # 'Keratinization',
    ],
    # 'Cellular Stress Response': [
    #     'Cellular response to heat stress',
    #     'Cellular response to hypoxia',
    #     'XBP1(S) activates chaperone genes',
    #     'NFE2L2 regulating anti-oxidant/detoxification enzymes',
    #     'KEAP1-NFE2L2 pathway',
    #     'Cachexia Signaling Pathway',
    #     'Response of EIF2AK1 (HRI) to heme deficiency',
    # ],
    # 'Hormone / Nuclear Receptor Signaling': [
    #     'ESR-mediated signaling',
    #     'Extra-nuclear estrogen signaling',
    #     'TR/RXR Activation',
    #     'VDR/RXR Activation',
    #     'Oxytocin Signaling Pathway',
    #     'Oxytocin in Brain Signaling Pathway',
    #     'α-Adrenergic Signaling',
    # ],
    # 'Growth Factor / RTK Signaling': [
    #     'IGF-1 Signaling',
    #     'Regulation of Insulin-like Growth Factor (IGF) transport and uptake by IGFBPs',
    #     'NGF Signaling',
    #     'MSP-RON Signaling in Cancer Cells Pathway',
    #     'Signaling by MET',
    #     'GP6 Signaling Pathway',
    #     'Thrombin Signaling',
    #     'Signaling by VEGF',
    # ],
    # 'Other': [
    #     'ABC-family proteins mediated transport',
    #     'ABRA Signaling Pathway',
    #     'Apoptotic execution phase',
    #     'Centrosomal KIAA0586 Signaling Pathway',
    #     'Cilia Biogenesis Signaling Pathway',
    #     'DYRK1A Signaling Pathway',
    #     'Epithelial Membrane Protein Signaling Pathway',
    #     'Folate Signaling Pathway',
    #     'G-Protein Coupled Receptor Signaling',
    #     'Glycation Signaling Pathway',
    #     'Hereditary Breast Cancer Signaling',
    #     'Interconversion of nucleotide di- and triphosphates',
    #     'L1CAM interactions',
    #     'Metabolism of polyamines',
    #     'Microautophagy Signaling Pathway',
    #     'Myelination Signaling Pathway',
    #     'Parkinson\'s Signaling Pathway',
    #     'Plasma lipoprotein assembly, remodeling, and clearance',
    #     'Regulation of Apoptosis',
    #     'Regulation of RUNX2 expression and activity',
    #     'Role of Tissue Factor in Cancer',
    #     'Sertoli Cell-Germ Cell Junction Signaling Pathway',
    #     'Sertoli Cell-Sertoli Cell Junction Signaling',
    #     'Signaling by Hippo',
    #     'Signaling by ROBO receptors',
    #     'Transcriptional regulation by RUNX1',
    #     'Transcriptional regulation by RUNX3',
    #     'Type II Diabetes Mellitus Signaling',
    # ],
})

story_6_genes = build_gene_groups(
    ipa_dict       = results_all_ipa,
    results        = results,
    comparisons    = comparisons,
    pathway_groups = pathway_groups,
    min_padj       = 0.05,
    min_lfc        = 0.3,
    remove_overlap = True,
)


fig, ax = gene_level_dotplot(
    results     = results,
    gene_groups = story_6_genes,
    comparisons = comparisons,
    # col_colors  = {
    #     "Fusion_vs_Control": "#FF8C00",
    #     # "MS_vs_Control":     "#5E80E5",
    #     # "CF_vs_Control":     "#1A8D8D"
    # },
    cl_labels   = {"1806": "HCC1806", "231": "MDA-MB-231"},
    title       = '  ',
    lfc_range   = 0.65,
    size_cap_p  = 1e-5,
    save_path   = 'plot_ecm.svg',
    show_group_labels=True,
    sort_by_significance = True,
    rank_method='mean_p_penalized',
    top_n                = 20,
    ylabel      = "ECM / Fibrosis / Collagen (Top Genes from Top Pathways)",
    col_spacing=0.35,
    row_height=0.32,
    size_legend_x=0.8,
    legend_font_sz=14,
)

ecm_table = gene_level_table(
    results     = results,
    gene_groups = story_6_genes,
    comparisons = comparisons,
    sort_by_significance = True,
    top_n                = None,
    save_path   = 'genes_ecm.tsv',
)
print(ecm_table)

### Metrics

In [ ]:
# Mitochondria concordance
pairwise_concordance(mito_table, 'C2C4', 'C6C8')

In [ ]:
# every pair at once, for picking the best example
all_pairwise_concordance(mito_table)   

In [ ]:
# every pair at once, for picking the best example
all_pairwise_concordance(interferon_table)   

In [ ]:
# every pair at once, for picking the best example
all_pairwise_concordance(proteo_table)

In [ ]:
# which clones are biggest responders across categories
gene_tables = {
    'Translation/RiBi': transl_table,
    'Cell Cycle':        cellcycle_table,
    'Proteostasis':      proteo_table,
    'Mitochondria':      mito_table,
    'Interferon':        interferon_table,
    'RNA Processing':    rnaproc_table,
    'ECM/Fibrosis':      ecm_table,
}
compare_sig_counts_across_groups(gene_tables)